In [89]:
import pandas as pd
import os
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
from sklearn.preprocessing import StandardScaler
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    log_loss,
    classification_report,
    confusion_matrix
)
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.base import clone
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.metrics import precision_score, recall_score
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.utils.validation import (
    check_X_y,
    check_array,
    check_is_fitted
)
from functools import partial


from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from imblearn import FunctionSampler
from imblearn.base import BaseSampler


from collections import Counter
from math import ceil

# Load & Inspect Master Dataset

In [2]:
start_train = 2016
end_train = 2024
start_validation = 2025
end_validation = 2025
start_test = 2026
end_test = 2026

In [3]:
file_path = r"D:\Project_Github\set_fear_index\dataset\verified_build_20261113_152948\SET50_Master_Dataset_DateSymbol_Sorted_VolZ_Fixed.parquet"

df = pd.read_parquet(file_path)

df

,Date,Period,Symbol,Open,High,Low,Close,Volume,daily_return,momentum_20d,vol_z,target,volatility_ret,VolRank,Corr_60d,CorrRank,LiquidityRank
0,2016-01-04,2016_H1,ADVANC,89.400711,89.700713,84.600673,85.800682,24880200.0,-0.059211,-0.302439,0.125602,2,0.054190,97.619048,NaN,NaN,100.000000
1,2016-01-04,2016_H1,AOT,31.625828,31.807062,30.719644,31.082115,30288000.0,-0.008670,0.075080,0.252794,2,0.016924,16.666667,NaN,NaN,90.476190
2,2016-01-04,2016_H1,BA,16.271213,16.981746,16.058053,16.910692,15762000.0,0.030303,0.048458,1.912703,0,0.019418,28.571429,NaN,NaN,38.095238
3,2016-01-04,2016_H1,BBL,97.331499,97.652725,96.367821,96.367821,2206700.0,-0.016394,-0.099099,-0.618234,2,0.020148,33.333333,NaN,NaN,85.714286
4,2016-01-04,2016_H1,BCP,19.514369,20.110228,19.365405,19.663334,2111200.0,0.000000,-0.057143,-0.513826,2,0.020182,35.714286,NaN,NaN,16.666667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
198527,2026-11-13,2026_H2,TTB,2.680000,2.740000,2.680000,2.720000,166939222.0,0.007407,-0.062069,-0.832242,2,0.017303,60.000000,0.454711,59.183673,68.000000
198528,2026-11-13,2026_H2,TTW,10.000000,10.100000,10.000000,10.100000,359531.0,0.010000,0.020202,-0.975033,2,0.007153,NaN,0.282357,NaN,NaN
198529,2026-11-13,2026_H2,TU,13.300000,13.400000,13.200000,13.400000,3070467.0,0.007519,0.063492,-1.478145,2,0.011891,20.000000,0.249204,8.163265,14.000000
198530,2026-11-13,2026_H2,VGI,0.950000,0.950000,0.920000,0.940000,71393873.0,-0.010526,-0.096154,-0.842176,2,0.047988,NaN,0.338034,NaN,NaN


In [4]:
df.shape

(198532, 17)

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 198532 entries, 0 to 198531
Data columns (total 17 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   Date            198532 non-null  datetime64[ms]
 1   Period          198532 non-null  str           
 2   Symbol          198532 non-null  string        
 3   Open            198532 non-null  float64       
 4   High            198532 non-null  float64       
 5   Low             198532 non-null  float64       
 6   Close           198532 non-null  float64       
 7   Volume          198532 non-null  float64       
 8   daily_return    198514 non-null  float64       
 9   momentum_20d    198168 non-null  float64       
 10  vol_z           197152 non-null  float64       
 11  target          198532 non-null  int64         
 12  volatility_ret  198168 non-null  float64       
 13  VolRank         119164 non-null  float64       
 14  Corr_60d        192654 non-null  float64       

In [6]:
df.describe()

,Date,Open,High,Low,Close,Volume,daily_return,momentum_20d,vol_z,target,volatility_ret,VolRank,Corr_60d,CorrRank,LiquidityRank
count,198532,198532.000000,198532.000000,198532.000000,198532.000000,1.985320e+05,198514.000000,198168.000000,197152.000000,198532.000000,198168.000000,119164.000000,192654.000000,116515.000000,119167.000000
mean,2021-08-27 08:05:13.194000,36.913794,37.315028,36.496896,36.892875,2.442942e+07,0.000349,0.007313,0.105467,1.871008,0.019347,51.106878,0.432264,51.106724,51.106850
min,2016-01-04 00:00:00,0.150000,0.160000,0.140000,0.140000,0.000000e+00,-0.420765,-0.795604,-4.312440,0.000000,0.000000,2.000000,-0.526897,2.000000,2.000000
25%,2018-12-28 00:00:00,7.415709,7.518018,7.311037,7.415708,3.272700e+06,-0.009434,-0.045802,-0.730526,2.000000,0.012239,26.086957,0.306570,26.086957,26.086957
50%,2021-10-06 00:00:00,19.560427,19.819911,19.274108,19.537790,8.873000e+06,0.000000,0.000000,-0.269665,2.000000,0.016867,51.162791,0.444819,51.162791,51.162791
75%,2024-05-13 00:00:00,42.214759,42.664354,41.662691,42.179429,2.240998e+07,0.009174,0.051429,0.466143,2.000000,0.023321,76.086957,0.571504,76.086957,76.086957
max,2026-11-13 00:00:00,403.302465,404.769019,398.902830,404.574951,2.473237e+09,2.162651,4.361446,932.252787,2.000000,0.487159,100.000000,0.936572,100.000000,100.000000
std,NaN,51.969600,52.408407,51.509252,51.945265,6.111343e+07,0.022879,0.108858,2.786974,0.446760,0.012108,28.860609,0.193401,28.860612,28.860609


In [7]:
df["vol_z"]

0         0.125602
1         0.252794
2         1.912703
3        -0.618234
4        -0.513826
            ...   
198527   -0.832242
198528   -0.975033
198529   -1.478145
198530   -0.842176
198531    0.716914
Name: vol_z, Length: 198532, dtype: float64

In [8]:
df["vol_z"] = df["vol_z"].replace([np.inf, -np.inf], np.nan)
print(
    "จำนวนค่า inf:",
    ((df["vol_z"] == np.inf) | (df["vol_z"] == -np.inf)).sum(),
)

จำนวนค่า inf: 0


In [9]:
dup_count = df.duplicated(subset=["Date", "Symbol"]).sum()
print(f"1. Duplicate Check (Date + Symbol): พบข้อมูลซ้ำ {dup_count} แถว")

if dup_count > 0:
  # แสดงตัวอย่างแถวที่ซ้ำ
  print(
      df[df.duplicated(subset=["Date", "Symbol"], keep=False)].head(
          10
      )
  )
else:
  print("ไม่มีข้อมูลซ้ำกัน")

1. Duplicate Check (Date + Symbol): พบข้อมูลซ้ำ 0 แถว
ไม่มีข้อมูลซ้ำกัน


In [10]:
missing_df = df.isnull().sum()
print("\n2. Missing Values Check:")
print(missing_df[missing_df > 0]) 

if missing_df.sum() == 0:
  print("   -> ผ่าน! ไม่พบ Missing Values ในข้อมูล")


2. Missing Values Check:
daily_return         18
momentum_20d        364
vol_z              1380
volatility_ret      364
VolRank           79368
Corr_60d           5878
CorrRank          82017
LiquidityRank     79365
dtype: int64


In [11]:
if "VolRank" in df.columns:
  print("\n3. VolRank Range Check:")
  print(df["VolRank"].describe()[["min", "max", "mean"]])
  out_of_range = df[
      (df["VolRank"] < 0) | (df["VolRank"] > 100)
  ]
  if len(out_of_range) > 0:
    print(
        f"พบ VolRank นอกช่วง 0-100 จำนวน {len(out_of_range)} แถว"
    )
  else:
    print("VolRank อยู่ในช่วง 0 ถึง 100 ทั้งหมด")
else:
  print("\n3. VolRank Range Check: ไม่พบคอลัมน์ VolRank ใน DataFrame")


3. VolRank Range Check:
min       2.000000
max     100.000000
mean     51.106878
Name: VolRank, dtype: float64
VolRank อยู่ในช่วง 0 ถึง 100 ทั้งหมด


# Target Alignment

In [12]:
# สร้างตารางวันซื้อขายถัดไปจากวันที่ที่มีอยู่จริง
trading_dates = pd.DatetimeIndex(
    sorted(df["Date"].dropna().unique())
)

date_mapping = pd.DataFrame({
    "Date": trading_dates[:-1],
    "Next_Trading_Date": trading_dates[1:]
})

date_mapping

,Date,Next_Trading_Date
0,2016-01-04,2016-01-05
1,2016-01-05,2016-01-06
2,2016-01-06,2016-01-07
3,2016-01-07,2016-01-08
4,2016-01-08,2016-01-11
...,...,...
2632,2026-11-06,2026-11-09
2633,2026-11-09,2026-11-10
2634,2026-11-10,2026-11-11
2635,2026-11-11,2026-11-12


In [13]:
df = df.merge(
    date_mapping,
    on="Date",
    how="left",
    validate="many_to_one"
)

df[["Date", "Next_Trading_Date", "Symbol"]].head()

,Date,Next_Trading_Date,Symbol
0,2016-01-04,2016-01-05,ADVANC
1,2016-01-04,2016-01-05,AOT
2,2016-01-04,2016-01-05,BA
3,2016-01-04,2016-01-05,BBL
4,2016-01-04,2016-01-05,BCP


In [14]:
future_df = df[
    ["Date", "Symbol", "daily_return", "vol_z"]
].copy()

future_df = future_df.rename(columns={
    "Date": "Next_Trading_Date",
    "daily_return": "future_return",
    "vol_z": "future_vol_z"
})

df = df.merge(
    future_df,
    on=["Next_Trading_Date", "Symbol"],
    how="left",
    validate="many_to_one"
)

In [15]:
# ดึงข้อมูล daily_return และ vol_z ของหุ้นตัวเดิมในวันซื้อขายถัดไป
future_df = df[
    ["Date", "Symbol", "daily_return", "vol_z"]
].copy()

future_df = future_df.rename(columns={
    "Date": "Next_Trading_Date",
    "daily_return": "future_return",
    "vol_z": "future_vol_z"
})

df = df.merge(
    future_df,
    on=["Next_Trading_Date", "Symbol"],
    how="left",
    validate="many_to_one"
)

$$
Y_{i,t+1} = \begin{cases} 
0 & R_{i,t+1} < 0,\ ZV_{i,t+1} \geq 1.5 \\ 
1 & R_{i,t+1} > 0,\ ZV_{i,t+1} \geq 1.5 \\ 
2 & \text{otherwise} 
\end{cases}
$$

โดยกำหนด:
- **0** = Panic
- **1** = FOMO
- **2** = Normal


In [16]:
# นำ daily_return และ vol_z ของวันซื้อขายถัดไปมาเชื่อมกับหุ้นตัวเดิม
future_df = df[
    ["Date", "Symbol", "daily_return", "vol_z"]
].copy()

future_df = future_df.rename(columns={
    "Date": "Next_Trading_Date",
    "daily_return": "future_return",
    "vol_z": "future_vol_z"
})

df = df.merge(
    future_df,
    on=["Next_Trading_Date", "Symbol"],
    how="left",
    validate="many_to_one"
)

In [17]:
print("Total rows:", len(df))

print("\nSample alignment:")
print(
    df.loc[
        df["Date"] == "2016-01-04",
        ["Date", "Next_Trading_Date", "Symbol",
         "future_return", "future_vol_z"]
    ].head(10)
)

print("\nMissing future data:")
print(
    df[["future_return", "future_vol_z"]].isna().sum()
)

Total rows: 198532

Sample alignment:
        Date Next_Trading_Date  Symbol  future_return  future_vol_z
0 2016-01-04        2016-01-05  ADVANC      -0.083916      1.383465
1 2016-01-04        2016-01-05     AOT      -0.008746      0.531222
2 2016-01-04        2016-01-05      BA      -0.012605      0.483912
3 2016-01-04        2016-01-05     BBL       0.026667     -0.322802
4 2016-01-04        2016-01-05     BCP       0.015152     -0.377642
5 2016-01-04        2016-01-05    BDMS       0.004546      0.462682
6 2016-01-04        2016-01-05  BEAUTY      -0.018349     -0.901651
7 2016-01-04        2016-01-05     BEC       0.032787      0.316941
8 2016-01-04        2016-01-05     BEM      -0.024504      8.110292
9 2016-01-04        2016-01-05      BH       0.048544      0.563362

Missing future data:
future_return      85
future_vol_z     1445
dtype: int64


In [18]:
# ตรวจสอบสาเหตุ Missing Future Data

missing_future = df[
    df["future_return"].isna() |
    df["future_vol_z"].isna()
].copy()

print("Total rows with missing future data:", len(missing_future))

print("\nMissing by column:")
print(df[["future_return", "future_vol_z"]].isna().sum())

print("\nMissing Next Trading Date:")
print(df["Next_Trading_Date"].isna().sum())

print("\nMissing breakdown:")
print(
    missing_future.assign(
        Missing_Return=missing_future["future_return"].isna(),
        Missing_Vol_Z=missing_future["future_vol_z"].isna()
    ).groupby(
        ["Missing_Return", "Missing_Vol_Z"],
        dropna=False
    ).size()
)

print("\nSample missing rows:")
print(
    missing_future[
        ["Date", "Next_Trading_Date", "Symbol",
         "future_return", "future_vol_z"]
    ].head(20)
)

Total rows with missing future data: 1445

Missing by column:
future_return      85
future_vol_z     1445
dtype: int64

Missing Next Trading Date:
83

Missing breakdown:
Missing_Return  Missing_Vol_Z
False           True             1360
True            True               85
dtype: int64

Sample missing rows:
            Date Next_Trading_Date Symbol  future_return  future_vol_z
50    2016-01-04        2016-01-05   SPRC       0.040000           NaN
115   2016-01-05        2016-01-06   SPRC       0.043956           NaN
180   2016-01-06        2016-01-07   SPRC      -0.005263           NaN
13003 2016-10-28        2016-10-31  BANPU       0.036697           NaN
13069 2016-10-31        2016-11-01  BANPU      -0.008850           NaN
13135 2016-11-01        2016-11-02  BANPU       0.017857           NaN
13201 2016-11-02        2016-11-03  BANPU      -0.017544           NaN
13267 2016-11-03        2016-11-04  BANPU      -0.008929           NaN
13333 2016-11-04        2016-11-07  BANPU       0.

In [19]:
# ตรวจสอบ Missing Return

missing_return = df[
    df["future_return"].isna()
].copy()

print("Total missing future_return:", len(missing_return))

print("\nMissing Next Trading Date:")
print(missing_return["Next_Trading_Date"].isna().sum())

print("\nMissing by Symbol:")
print(
    missing_return.groupby("Symbol")
    .size()
    .sort_values(ascending=False)
    .head(20)
)

print("\nRows with Next Trading Date but missing Return:")

print(
    missing_return[
        missing_return["Next_Trading_Date"].notna()
    ][
        ["Date", "Next_Trading_Date", "Symbol"]
    ].to_string(index=False)
)

Total missing future_return: 85

Missing Next Trading Date:
83

Missing by Symbol:
Symbol
BANPU     2
SCB       2
AOT       1
AWC       1
BA        1
BAM       1
BBL       1
BCP       1
BDMS      1
BEAUTY    1
BEC       1
BEM       1
BGRIM     1
BH        1
BJC       1
BLA       1
BTS       1
CBG       1
ADVANC    1
CCET      1
dtype: int64

Rows with Next Trading Date but missing Return:
      Date Next_Trading_Date Symbol
2022-04-20        2022-04-21    SCB
2026-08-11        2026-08-13  BANPU


In [20]:
#ใช้ future_return และ future_vol_z ที่เรียงเรียบร้อยแล้ว โดยไม่เปลี่ยนนิยาม Target เดิม

conditions = [
    (df["future_return"] < 0) & (df["future_vol_z"] >= 1.5),
    (df["future_return"] > 0) & (df["future_vol_z"] >= 1.5)
]

choices = [0, 1]

df["target_aligned"] = np.select(
    conditions,
    choices,
    default=2
)

# Missing future data ต้องไม่มี Target
missing_target = (
    df["future_return"].isna() |
    df["future_vol_z"].isna()
)

df.loc[missing_target, "target_aligned"] = np.nan

In [21]:
print("Target Distribution:")
print(
    df["target_aligned"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nTarget Percentage:")
print(
    df["target_aligned"]
    .value_counts(normalize=True)
    .sort_index() * 100
)

print("\nMissing Target:")
print(df["target_aligned"].isna().sum())

Target Distribution:
target_aligned
0.0      8098
1.0     11031
2.0    177958
NaN      1445
Name: count, dtype: int64

Target Percentage:
target_aligned
0.0     4.108845
1.0     5.597021
2.0    90.294134
Name: proportion, dtype: float64

Missing Target:
1445


In [22]:
# ตรวจสอบว่า Target ตรงตามเงื่อนไขหรือไม่

print("Panic violations:",
    ((df["target_aligned"] == 0) &
     ~((df["future_return"] < 0) &
       (df["future_vol_z"] >= 1.5))).sum()
)

print("FOMO violations:",
    ((df["target_aligned"] == 1) &
     ~((df["future_return"] > 0) &
       (df["future_vol_z"] >= 1.5))).sum()
)

print("Normal violations:",
    ((df["target_aligned"] == 2) &
     ((df["future_return"].isna()) |
      (df["future_vol_z"].isna()))).sum()
)

Panic violations: 0
FOMO violations: 0
Normal violations: 0


In [23]:
comparison = df[
    ["target", "target_aligned"]
].copy()

comparison["same"] = (
    comparison["target"] == comparison["target_aligned"]
)

print("Comparison:")
print(comparison["same"].value_counts(dropna=False))

print("\nOld Target Distribution:")
print(df["target"].value_counts().sort_index())

print("\nNew Target Distribution:")
print(df["target_aligned"].value_counts(dropna=False).sort_index())

Comparison:
same
True     170554
False     27978
Name: count, dtype: int64

Old Target Distribution:
target
0      8660
1      8289
2    181583
Name: count, dtype: int64

New Target Distribution:
target_aligned
0.0      8098
1.0     11031
2.0    177958
NaN      1445
Name: count, dtype: int64


In [24]:
comparison_table = pd.crosstab(
    df["target"],
    df["target_aligned"],
    rownames=["Old Target"],
    colnames=["New Target"],
    dropna=False
)

print(comparison_table)

New Target   0.0   1.0     2.0   NaN
Old Target                          
0           2118     0    6541     1
1              0  2654    5635     0
2           5980  8377  165782  1444


# Model

In [25]:
features = [
    "daily_return",
    "momentum_20d",
    "vol_z",
    "VolRank",
    "CorrRank",
    "LiquidityRank"
]

target_col = "target_aligned"

In [26]:
# เอา SET50 ในแต่ละ period
constituents = pd.read_csv(
    r"D:\Project_Github\set_fear_index\dataset\SET50_constituents_by_halfyear_2016_2026.csv"
)

# Convert constituents into long format
symbol_cols = [col for col in constituents.columns if col.startswith("symbol_")]

constituents_long = constituents.melt(
    id_vars="period",
    value_vars=symbol_cols,
    value_name="Symbol"
)

constituents_long = constituents_long.dropna(subset=["Symbol"])

constituents_long["Period"] = constituents_long["period"].astype(str)
constituents_long["Symbol"] = constituents_long["Symbol"].astype(str).str.strip().str.upper()

# Normalize master dataset
df["Symbol"] = df["Symbol"].astype(str).str.strip().str.upper()

# Check membership on predictor date t
membership = constituents_long[["Period", "Symbol"]].drop_duplicates()

df = df.merge(
    membership.assign(is_member_t=True),
    on=["Period", "Symbol"],
    how="left",
    validate="many_to_one"
)

df["is_member_t"] = df["is_member_t"].fillna(False).astype(bool)

print("Total rows:", len(df))
print("SET50 member rows:", df["is_member_t"].sum())
print("Non-member rows:", (~df["is_member_t"]).sum())

Total rows: 198532
SET50 member rows: 119224
Non-member rows: 79308


In [27]:
# ตรวจสอบ Membership Merge
print("Duplicate Date-Symbol:",
      df.duplicated(["Date", "Symbol"]).sum())

print("Membership missing:",
      df["is_member_t"].isna().sum())

print("\nMembership by Period:")
print(
    df.groupby("Period")["is_member_t"]
      .agg(["sum", "count"])
)

Duplicate Date-Symbol: 0
Membership missing: 0

Membership by Period:
          sum  count
Period              
2016_H1  5040   7800
2016_H2  5128   8124
2017_H1  5160   8096
2017_H2  5124   8460
2018_H1  5246   8540
2018_H2  5289   8660
2019_H1  5040   8520
2019_H2  5332   8869
2020_H1  5412   9068
2020_H2  5280   9045
2021_H1  5310   9058
2021_H2  5658   9471
2022_H1  5399   9207
2022_H2  5612   9638
2023_H1  5566   9680
2023_H2  5734   9760
2024_H1  5712   9520
2024_H2  5828   9920
2025_H1  5767   9605
2025_H2  6150  10124
2026_H1  5900   9794
2026_H2  4537   7573


In [28]:
# Modeling Dataset: preserve Period for fold-level diagnostics, not as a model feature
model_df = df.loc[
    df["is_member_t"],
    ["Date", "Next_Trading_Date", "Period", "Symbol"] + features + [target_col]
].copy()

model_df = model_df.rename(columns={
    "Next_Trading_Date": "Target_Date"
})

print("Modeling Dataset shape:", model_df.shape)
print("Periods:", model_df["Period"].nunique())
print("Features and target remain unchanged:", features, target_col)

print("\nMissing values:")
print(model_df.isna().sum())

print("\nRows with missing target:",
      model_df[target_col].isna().sum())

print("Rows with missing features:",
      model_df[features].isna().any(axis=1).sum())

Modeling Dataset shape: (119224, 11)
Periods: 22
Features and target remain unchanged: ['daily_return', 'momentum_20d', 'vol_z', 'VolRank', 'CorrRank', 'LiquidityRank'] target_aligned

Missing values:
Date                 0
Target_Date         50
Period               0
Symbol               0
daily_return         3
momentum_20d        60
vol_z               60
VolRank             60
CorrRank          2709
LiquidityRank       57
target_aligned     108
dtype: int64

Rows with missing target: 108
Rows with missing features: 2709


In [29]:
# ตรวจสอบสาเหตุ Missing Values
print("Target_Date missing:",
      model_df["Target_Date"].isna().sum())

print("\nTarget missing breakdown:")
print(
    df.loc[df["is_member_t"], [
        "Next_Trading_Date",
        "future_return",
        "future_vol_z",
        "target_aligned"
    ]].isna().sum()
)

# Missing features
print("\nMissing by feature:")
print(model_df[features].isna().sum())

# CorrRank vs Corr_60d
member_df = df.loc[df["is_member_t"]].copy()

print("\nCorrRank missing:", member_df["CorrRank"].isna().sum())
print("Corr_60d missing:", member_df["Corr_60d"].isna().sum())

print("\nCorrRank missing by Period:")
print(
    member_df.groupby("Period")["CorrRank"]
    .apply(lambda x: x.isna().sum())
)

print("\nSample rows with missing CorrRank:")
print(
    member_df.loc[
        member_df["CorrRank"].isna(),
        ["Date", "Period", "Symbol", "Corr_60d"]
    ].head(15)
)

Target_Date missing: 50

Target missing breakdown:
Next_Trading_Date     50
future_return         52
future_vol_z         108
target_aligned       108
dtype: int64

Missing by feature:
daily_return        3
momentum_20d       60
vol_z              60
VolRank            60
CorrRank         2709
LiquidityRank      57
dtype: int64

CorrRank missing: 2709
Corr_60d missing: 2709

CorrRank missing by Period:
Period
2016_H1    2478
2016_H2      44
2017_H1      16
2017_H2       0
2018_H1       0
2018_H2       0
2019_H1       0
2019_H2       0
2020_H1       6
2020_H2       0
2021_H1      14
2021_H2       0
2022_H1      44
2022_H2      16
2023_H1       0
2023_H2       0
2024_H1       0
2024_H2       0
2025_H1      55
2025_H2      35
2026_H1       0
2026_H2       1
Name: CorrRank, dtype: int64

Sample rows with missing CorrRank:
         Date   Period  Symbol  Corr_60d
0  2016-01-04  2016_H1  ADVANC       NaN
1  2016-01-04  2016_H1     AOT       NaN
2  2016-01-04  2016_H1      BA       NaN
3  201

In [30]:
# รูปแบบ Target missing
target_missing = df.loc[
    df["is_member_t"] & df["target_aligned"].isna()
].copy()

print("Target missing pattern:")
print(pd.crosstab(
    target_missing["Next_Trading_Date"].isna(),
    target_missing["future_return"].isna()
))

print("\nTarget missing by Period:")
print(target_missing.groupby("Period").size())

# CorrRank missing นอกจาก 2016_H1
corr_missing = member_df.loc[
    member_df["CorrRank"].isna()
].copy()

print("\nCorrRank missing outside 2016_H1:")
print(
    corr_missing.loc[
        corr_missing["Period"] != "2016_H1"
    ].groupby("Period").agg(
        rows=("Symbol", "size"),
        first_date=("Date", "min"),
        last_date=("Date", "max"),
        symbols=("Symbol", "nunique")
    )
)

# 3. Feature missing combinations
missing_features = model_df[features].isna()

missing_pattern = (
    missing_features.apply(
        lambda row: ", ".join(row.index[row].tolist()),
        axis=1
    )
)

print("\nFeature missing combinations:")
print(missing_pattern.value_counts())

# 4. Inspect daily_return missing
print("\nRows with missing daily_return:")
print(
    model_df.loc[
        model_df["daily_return"].isna(),
        ["Date", "Target_Date", "Symbol", "daily_return"]
    ]
)

Target missing pattern:
future_return      False  True 
Next_Trading_Date              
False                 56      2
True                   0     50

Target missing by Period:
Period
2016_H2    19
2022_H1    19
2025_H1    19
2026_H2    51
dtype: int64

CorrRank missing outside 2016_H1:
         rows first_date  last_date  symbols
Period                                      
2016_H2    44 2016-10-28 2016-12-30        1
2017_H1    16 2017-01-04 2017-01-26        1
2020_H1     6 2020-01-02 2020-01-09        1
2021_H1    14 2021-01-04 2021-01-21        1
2022_H1    44 2022-04-20 2022-06-30        1
2022_H2    16 2022-07-01 2022-07-25        1
2025_H1    55 2025-04-03 2025-06-30        1
2025_H2    35 2025-07-01 2025-08-15        2
2026_H2     1 2026-11-13 2026-11-13        1

Feature missing combinations:
                                                                       116515
CorrRank                                                                 2649
momentum_20d, vol_z, VolRank

In [31]:
#เจาะจงหุ้นที่ CorrRank หาย
print("CorrRank missing by Period and Symbol:")

print(
    corr_missing.groupby(["Period", "Symbol"]).agg(
        rows=("Date", "size"),
        first_date=("Date", "min"),
        last_date=("Date", "max")
    ).to_string()
)

CorrRank missing by Period and Symbol:
                rows first_date  last_date
Period  Symbol                            
2016_H1 ADVANC    59 2016-01-04 2016-03-25
        AOT       59 2016-01-04 2016-03-25
        BA        59 2016-01-04 2016-03-25
        BBL       59 2016-01-04 2016-03-25
        BCP       59 2016-01-04 2016-03-25
        BDMS      59 2016-01-04 2016-03-25
        BEC       59 2016-01-04 2016-03-25
        BH        59 2016-01-04 2016-03-25
        BLA       59 2016-01-04 2016-03-25
        BTS       59 2016-01-04 2016-03-25
        CBG       59 2016-01-04 2016-03-25
        CENTEL    59 2016-01-04 2016-03-25
        CK        59 2016-01-04 2016-03-25
        CPALL     59 2016-01-04 2016-03-25
        CPF       59 2016-01-04 2016-03-25
        CPN       59 2016-01-04 2016-03-25
        DELTA     59 2016-01-04 2016-03-25
        EGCO      59 2016-01-04 2016-03-25
        HMPRO     59 2016-01-04 2016-03-25
        IRPC      59 2016-01-04 2016-03-25
        ITD    

In [32]:
missing_return = model_df.loc[
    model_df["daily_return"].isna(),
    ["Date", "Symbol"]
]

for _, row in missing_return.iterrows():
    symbol = row["Symbol"]
    date = row["Date"]

    print(f"\n--- {symbol} | {date.date()} ---")

    history = df.loc[
        (df["Symbol"] == symbol) &
        (df["Date"].between(
            date - pd.Timedelta(days=5),
            date + pd.Timedelta(days=5)
        )),
        ["Date", "Symbol", "Open", "High", "Low",
         "Close", "Volume", "daily_return"]
    ]

    print(history.to_string(index=False))


--- BANPU | 2016-10-28 ---
      Date Symbol      Open      High       Low     Close      Volume  daily_return
2016-10-28  BANPU 16.669773 18.243025 16.535880 18.243025 388183300.0           NaN
2016-10-31  BANPU 18.410395 19.247231 17.908293 18.912497 134347200.0      0.036697
2016-11-01  BANPU 18.912494 19.247228 18.577760 18.745127  81274200.0     -0.008850
2016-11-02  BANPU 18.912493 19.414595 18.745126 19.079861  86056600.0      0.017857

--- SCB | 2022-04-20 ---
      Date Symbol      Open      High       Low     Close    Volume  daily_return
2022-04-20    SCB 48.255465 62.270839 46.836187 57.835594 2840200.0           NaN

--- GULF | 2025-04-03 ---
      Date Symbol      Open      High       Low     Close     Volume  daily_return
2025-04-03   GULF 46.668348 47.852823 46.194557 46.194557 86492700.0           NaN
2025-04-04   GULF 45.483870 45.720766 42.641129 42.641129 90056800.0     -0.076923
2025-04-08   GULF 36.955644 39.561491 36.481854 37.903225 97264300.0     -0.111111


In [33]:
check_symbols = {
    "BANPU": ["2016-10-28", "2017-01-26"],
    "AWC": ["2020-01-02", "2020-01-09"],
    "SCGP": ["2021-01-04", "2021-01-21"],
    "SCB": ["2022-04-20", "2022-07-25"],
    "GULF": ["2025-04-03", "2025-07-07"],
    "TIDLOR": ["2025-07-01", "2025-08-15"]
}

for symbol, dates in check_symbols.items():

    start = pd.Timestamp(dates[0])
    end = pd.Timestamp(dates[1])

    temp = df.loc[
        (df["Symbol"] == symbol) &
        (df["Date"].between(
            start - pd.Timedelta(days=10),
            end + pd.Timedelta(days=10)
        )),
        ["Date", "Symbol", "daily_return", "Corr_60d", "CorrRank"]
    ].sort_values("Date")

    print(f"\n===== {symbol} =====")
    print(temp.to_string(index=False))


===== BANPU =====
      Date Symbol  daily_return  Corr_60d  CorrRank
2016-10-28  BANPU           NaN       NaN       NaN
2016-10-31  BANPU      0.036697       NaN       NaN
2016-11-01  BANPU     -0.008850       NaN       NaN
2016-11-02  BANPU      0.017857       NaN       NaN
2016-11-03  BANPU     -0.017544       NaN       NaN
2016-11-04  BANPU     -0.008929       NaN       NaN
2016-11-07  BANPU      0.009009       NaN       NaN
2016-11-08  BANPU     -0.044643       NaN       NaN
2016-11-09  BANPU      0.046729       NaN       NaN
2016-11-10  BANPU     -0.017857       NaN       NaN
2016-11-11  BANPU      0.000000       NaN       NaN
2016-11-14  BANPU     -0.045455       NaN       NaN
2016-11-15  BANPU     -0.028571       NaN       NaN
2016-11-16  BANPU     -0.019608       NaN       NaN
2016-11-17  BANPU     -0.032000       NaN       NaN
2016-11-18  BANPU      0.064050       NaN       NaN
2016-11-21  BANPU     -0.029126       NaN       NaN
2016-11-22  BANPU      0.020000       NaN    

In [34]:
# Check missing values before filtering
missing_target = model_df[target_col].isna()
missing_features = model_df[features].isna().any(axis=1)

# Count exclusion categories
print("Total candidate rows:", len(model_df))
print("Missing target:", missing_target.sum())
print("Missing features:", missing_features.sum())
print("Missing both:", (missing_target & missing_features).sum())

# Keep rows with complete target and features
final_model_df = model_df.loc[
    (~missing_target) & (~missing_features)
].copy()

# Convert target to integer
final_model_df[target_col] = (
    final_model_df[target_col].astype(int)
)

print("\nFinal Modeling Dataset:", final_model_df.shape)
print("Remaining missing values:", final_model_df.isna().sum().sum())

print("\nTarget distribution:")
print(final_model_df[target_col].value_counts().sort_index())

Total candidate rows: 119224
Missing target: 108
Missing features: 2709
Missing both: 58

Final Modeling Dataset: (116465, 11)
Remaining missing values: 0

Target distribution:
target_aligned
0      5044
1      6183
2    105238
Name: count, dtype: int64


In [35]:
# ตรวจสอบช่วงเวลาของชุดข้อมูล
print("Feature Date Range:")
print(final_model_df["Date"].min(), "to", final_model_df["Date"].max())

print("\nTarget Date Range:")
print(final_model_df["Target_Date"].min(), "to", final_model_df["Target_Date"].max())

# ตรวจสอบจำนวนตัวอย่างรายปีตาม Target Date
final_model_df["Target_Year"] = final_model_df["Target_Date"].dt.year

print("\nSamples by Target Year:")
print(final_model_df["Target_Year"].value_counts().sort_index())

Feature Date Range:
2016-03-28 00:00:00 to 2026-11-12 00:00:00

Target Date Range:
2016-03-29 00:00:00 to 2026-11-13 00:00:00

Samples by Target Year:
Target_Year
2016     7605
2017    10267
2018    10534
2019    10372
2020    10685
2021    10952
2022    10951
2023    11299
2024    11540
2025    11824
2026    10436
Name: count, dtype: int64


In [36]:
# แบ่งข้อมูลตามปีของ Target Date
train_df = final_model_df[
    final_model_df["Target_Date"].dt.year.between(
        start_train, end_train
    )
].copy()

validation_df = final_model_df[
    final_model_df["Target_Date"].dt.year.between(
        start_validation, end_validation
    )
].copy()

test_df = final_model_df[
    final_model_df["Target_Date"].dt.year.between(
        start_test, end_test
    )
].copy()


# ตรวจสอบขนาดของแต่ละชุดข้อมูล
print("Train:", train_df.shape)
print("Validation:", validation_df.shape)
print("Test:", test_df.shape)

print("\nTotal samples:",
      len(train_df) + len(validation_df) + len(test_df))


# ตรวจสอบช่วงเวลาของแต่ละชุดข้อมูล
for name, data in [
    ("Train", train_df),
    ("Validation", validation_df),
    ("Test", test_df)
]:
    print(f"\n{name} Date Range:")
    print("Feature Date:", data["Date"].min(), "to", data["Date"].max())
    print("Target Date:", data["Target_Date"].min(), "to", data["Target_Date"].max())


# ตรวจสอบการกระจายของ Target
for name, data in [
    ("Train", train_df),
    ("Validation", validation_df),
    ("Test", test_df)
]:
    print(f"\n{name} Target Distribution:")
    print(data["target_aligned"].value_counts().sort_index())

Train: (94205, 12)
Validation: (11824, 12)
Test: (10436, 12)

Total samples: 116465

Train Date Range:
Feature Date: 2016-03-28 00:00:00 to 2024-12-27 00:00:00
Target Date: 2016-03-29 00:00:00 to 2024-12-30 00:00:00

Validation Date Range:
Feature Date: 2024-12-30 00:00:00 to 2025-12-29 00:00:00
Target Date: 2025-01-02 00:00:00 to 2025-12-30 00:00:00

Test Date Range:
Feature Date: 2025-12-30 00:00:00 to 2026-11-12 00:00:00
Target Date: 2026-01-05 00:00:00 to 2026-11-13 00:00:00

Train Target Distribution:
target_aligned
0     4114
1     4877
2    85214
Name: count, dtype: int64

Validation Target Distribution:
target_aligned
0      544
1      617
2    10663
Name: count, dtype: int64

Test Target Distribution:
target_aligned
0     386
1     689
2    9361
Name: count, dtype: int64


In [37]:
# แยก Features และ Target
X_train = train_df[features].copy()
y_train = train_df[target_col].copy()

X_val = validation_df[features].copy()
y_val = validation_df[target_col].copy()

X_test = test_df[features].copy()
y_test = test_df[target_col].copy()


# ตรวจสอบค่า Infinity และ Missing Values
for name, X in [
    ("Train", X_train),
    ("Validation", X_val),
    ("Test", X_test)
]:
    print(f"{name}:")
    print("Missing:", X.isna().sum().sum())
    print("Infinite:", np.isinf(X.to_numpy()).sum())

Train:
Missing: 0
Infinite: 0
Validation:
Missing: 0
Infinite: 0
Test:
Missing: 0
Infinite: 0


In [38]:
# สร้าง Scaler
scaler = StandardScaler()

# Fit และ Transform ด้วย Train
X_train_scaled = scaler.fit_transform(X_train)

# Transform Validation และ Test ด้วย Scaler เดิม
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)


# ตรวจสอบค่าเฉลี่ยและส่วนเบี่ยงเบนมาตรฐานของ Train
print("Train Mean:")
print(np.round(X_train_scaled.mean(axis=0), 4))

print("\nTrain Standard Deviation:")
print(np.round(X_train_scaled.std(axis=0), 4))

# ตรวจสอบขนาดข้อมูล
print("\nScaled Dataset Shapes:")
print("Train:", X_train_scaled.shape)
print("Validation:", X_val_scaled.shape)
print("Test:", X_test_scaled.shape)

Train Mean:
[ 0.  0.  0.  0. -0.  0.]

Train Standard Deviation:
[1. 1. 1. 1. 1. 1.]

Scaled Dataset Shapes:
Train: (94205, 6)
Validation: (11824, 6)
Test: (10436, 6)


Model A : Baseline
จุดอ้างอิง
- Softmax Logistic Regression
- class_weight=None
- ไม่ทำ Oversampling หรือ Undersampling
- ใช้สัดส่วนข้อมูลจริง

Model B : Balanced
ทดลองเพิ่มน้ำหนัก
- Softmax Logistic Regression
- class_weight="balanced"
- ไม่เปลี่ยนจำนวนแถวใน Dataset

Model C : Custom Class Weight
ทดลองเชิง Quant
- กำหนดน้ำหนัก Panic, FOMO และ Normal เอง
- ทดลองปรับระดับ Penalty
- เลือกน้ำหนักจากผล Validation

In [39]:
# สร้าง Softmax Logistic Regression
model_A = LogisticRegression(
    C=1.0,
    solver="lbfgs",
    l1_ratio=0,
    class_weight=None,
    max_iter=2000,
    random_state=42
)
# ฝึกโมเดลด้วย Train
model_A.fit(X_train_scaled, y_train)

# ตรวจสอบผลการฝึก
print("Model A Training Status")
print("Converged:", model_A.n_iter_)
print("Classes:", model_A.classes_)
print("Coefficients Shape:", model_A.coef_.shape)
print("Intercept Shape:", model_A.intercept_.shape)

Model A Training Status
Converged: [9]
Classes: [0 1 2]
Coefficients Shape: (3, 6)
Intercept Shape: (3,)


In [40]:
# ทำนาย Class และ Probability
y_val_pred_A = model_A.predict(X_val_scaled)
y_val_proba_A = model_A.predict_proba(X_val_scaled)

# Classification Metrics
print("Accuracy:", accuracy_score(y_val, y_val_pred_A))
print("Macro F1:", f1_score(y_val, y_val_pred_A, average="macro"))

print("\nClassification Report:")
print(classification_report(
    y_val,
    y_val_pred_A,
    labels=[0, 1, 2],
    target_names=["Panic", "FOMO", "Normal"],
    zero_division=0
))

# Probability Metrics
print("\nLog Loss:", log_loss(y_val, y_val_proba_A, labels=[0, 1, 2]))

# Confusion Matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_val_pred_A, labels=[0, 1, 2]))

Accuracy: 0.9011332882273342
Macro F1: 0.3264659828934468

Classification Report:
              precision    recall  f1-score   support

       Panic       0.44      0.01      0.03       544
        FOMO       0.17      0.00      0.01       617
      Normal       0.90      1.00      0.95     10663

    accuracy                           0.90     11824
   macro avg       0.50      0.34      0.33     11824
weighted avg       0.84      0.90      0.86     11824


Log Loss: 0.3760071111323278

Confusion Matrix:
[[    7     0   537]
 [    2     2   613]
 [    7    10 10646]]


- Panic เกิดขึ้นจริง 544 ตัวอย่าง แต่ตรวจจับได้เพียง 7 ตัวอย่าง
- FOMO เกิดขึ้นจริง 617 ตัวอย่าง แต่ตรวจจับได้เพียง 2 ตัวอย่าง
- Normal ถูกทำนายถูกต้อง 10,646 จาก 10,663 ตัวอย่าง

In [41]:
# สร้าง Softmax Logistic Regression แบบถ่วงน้ำหนัก
model_B = LogisticRegression(
    C=1.0,
    solver="lbfgs",
    l1_ratio=0,
    class_weight="balanced",
    max_iter=2000,
    random_state=42
)

# ฝึกโมเดลด้วย Train
model_B.fit(X_train_scaled, y_train)

# ตรวจสอบน้ำหนักของแต่ละ Class
print("Class Weights:")
print(model_B.class_weight)

# ตรวจสอบจำนวน Iterations
print("\nIterations:", model_B.n_iter_)

# ตรวจสอบ Classes
print("Classes:", model_B.classes_)

Class Weights:
balanced

Iterations: [6]
Classes: [0 1 2]


In [42]:
# ทำนาย Class และ Probability
y_val_pred_B = model_B.predict(X_val_scaled)
y_val_proba_B = model_B.predict_proba(X_val_scaled)

# Classification Metrics
print("Accuracy:", accuracy_score(y_val, y_val_pred_B))
print("Macro F1:", f1_score(y_val, y_val_pred_B, average="macro"))

print("\nClassification Report:")
print(classification_report(
    y_val,
    y_val_pred_B,
    labels=[0, 1, 2],
    target_names=["Panic", "FOMO", "Normal"],
    zero_division=0
))

# Probability Metrics
print("\nLog Loss:", log_loss(y_val, y_val_proba_B, labels=[0, 1, 2]))

# Confusion Matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_val_pred_B, labels=[0, 1, 2]))

Accuracy: 0.5675744248985115
Macro F1: 0.3327356937702519

Classification Report:
              precision    recall  f1-score   support

       Panic       0.08      0.43      0.14       544
        FOMO       0.09      0.32      0.14       617
      Normal       0.94      0.59      0.72     10663

    accuracy                           0.57     11824
   macro avg       0.37      0.45      0.33     11824
weighted avg       0.85      0.57      0.67     11824


Log Loss: 1.0467672575108236

Confusion Matrix:
[[ 236  116  192]
 [ 187  196  234]
 [2463 1921 6279]]


In [43]:
# กำหนดน้ำหนักของแต่ละ Class
custom_weights = {
    0: 2.0,  # Panic
    1: 2.0,  # FOMO
    2: 1.0   # Normal
}

# สร้าง Softmax Logistic Regression
model_C = LogisticRegression(
    C=1.0,
    solver="lbfgs",
    l1_ratio=0,
    class_weight=custom_weights,
    max_iter=2000,
    random_state=42
)

# ฝึกโมเดล
model_C.fit(X_train_scaled, y_train)

# ตรวจสอบผลการฝึก
print("Iterations:", model_C.n_iter_)
print("Classes:", model_C.classes_)
print("Custom Weights:", custom_weights)

Iterations: [8]
Classes: [0 1 2]
Custom Weights: {0: 2.0, 1: 2.0, 2: 1.0}


In [44]:
# ทำนาย Class และ Probability
y_val_pred_C = model_C.predict(X_val_scaled)
y_val_proba_C = model_C.predict_proba(X_val_scaled)

# Classification Metrics
print("Accuracy:", accuracy_score(y_val, y_val_pred_C))
print("Macro F1:", f1_score(y_val, y_val_pred_C, average="macro"))

print("\nClassification Report:")
print(classification_report(
    y_val,
    y_val_pred_C,
    labels=[0, 1, 2],
    target_names=["Panic", "FOMO", "Normal"],
    zero_division=0
))

# Probability Metrics
print("\nLog Loss:", log_loss(y_val, y_val_proba_C, labels=[0, 1, 2]))

# Confusion Matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_val_pred_C, labels=[0, 1, 2]))

Accuracy: 0.8996109607577808
Macro F1: 0.33248641376097604

Classification Report:
              precision    recall  f1-score   support

       Panic       0.30      0.02      0.03       544
        FOMO       0.18      0.01      0.02       617
      Normal       0.90      1.00      0.95     10663

    accuracy                           0.90     11824
   macro avg       0.46      0.34      0.33     11824
weighted avg       0.84      0.90      0.86     11824


Log Loss: 0.3993687521943892

Confusion Matrix:
[[   10     2   532]
 [    3     5   609]
 [   20    21 10622]]


# Temporal Cross-Validation Folds + Grid Search

In [45]:
# เรียงข้อมูลตาม Target Date และ Symbol
train_cv_df = (
    train_df
    .sort_values(["Target_Date", "Symbol"])
    .reset_index(drop=True)
)

# สร้างชุดวันที่ไม่ซ้ำ
unique_dates = np.array(
    sorted(train_cv_df["Target_Date"].unique())
)

# สร้าง Expanding Window Cross-Validation
tscv = TimeSeriesSplit(n_splits=5)

# เก็บตำแหน่งข้อมูลของแต่ละ Fold
cv_splits = []

# สร้างตารางตรวจสอบ Fold
fold_summary = []

for fold, (train_date_idx, val_date_idx) in enumerate(
    tscv.split(unique_dates), start=1
):

    # เลือกวันที่ของ Training และ Validation
    fold_train_dates = unique_dates[train_date_idx]
    fold_val_dates = unique_dates[val_date_idx]

    # แปลงวันที่เป็นตำแหน่งแถวใน Dataset
    fold_train_idx = np.flatnonzero(
        train_cv_df["Target_Date"]
        .isin(fold_train_dates)
        .to_numpy()
    )

    fold_val_idx = np.flatnonzero(
        train_cv_df["Target_Date"]
        .isin(fold_val_dates)
        .to_numpy()
    )

    # เก็บตำแหน่งแถวสำหรับใช้ใน Grid Search
    cv_splits.append((fold_train_idx, fold_val_idx))

    # เก็บข้อมูลสรุปของ Fold
    fold_summary.append({
        "Fold": fold,
        "Train_Start": pd.Timestamp(
            fold_train_dates[0]
        ).date(),
        "Train_End": pd.Timestamp(
            fold_train_dates[-1]
        ).date(),
        "Val_Start": pd.Timestamp(
            fold_val_dates[0]
        ).date(),
        "Val_End": pd.Timestamp(
            fold_val_dates[-1]
        ).date(),
        "Train_Rows": len(fold_train_idx),
        "Val_Rows": len(fold_val_idx)
    })

# แสดงผลการแบ่ง Fold
fold_summary_df = pd.DataFrame(fold_summary)

print("จำนวน Target Dates:", len(unique_dates))
print("จำนวน Folds:", len(cv_splits))
print()
print(fold_summary_df.to_string(index=False))

# ตรวจสอบว่าไม่มี Target Date ซ้ำข้าม Train และ Validation
for fold, (train_idx, val_idx) in enumerate(cv_splits, start=1):

    train_dates = set(
        train_cv_df.iloc[train_idx]["Target_Date"]
    )

    val_dates = set(
        train_cv_df.iloc[val_idx]["Target_Date"]
    )

    assert train_dates.isdisjoint(val_dates)

    assert max(train_dates) < min(val_dates)

print("\nตรวจสอบ Temporal Folds ผ่านทั้งหมด")


จำนวน Target Dates: 2126
จำนวน Folds: 5

 Fold Train_Start  Train_End  Val_Start    Val_End  Train_Rows  Val_Rows
    1  2016-03-29 2017-09-15 2017-09-18 2019-02-26       14932     15113
    2  2016-03-29 2019-02-26 2019-02-27 2020-08-10       30045     15282
    3  2016-03-29 2020-08-10 2020-08-11 2022-01-28       45327     15944
    4  2016-03-29 2022-01-28 2022-01-31 2023-07-17       61271     16177
    5  2016-03-29 2023-07-17 2023-07-18 2024-12-30       77448     16757

ตรวจสอบ Temporal Folds ผ่านทั้งหมด


In [46]:
# สร้าง Pipeline สำหรับ Scaling และ Softmax
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        solver="saga",
        max_iter=5000,
        random_state=42
    ))
])

# กำหนด Hyperparameter Grid
param_grid = {
    "model__C": [0.01, 0.1, 1, 10, 100],
    "model__l1_ratio": [0, 0.5, 1],
    "model__class_weight": [
        None,
        "balanced",
        {0: 2, 1: 2, 2: 1},
        {0: 5, 1: 5, 2: 1},
        {0: 10, 1: 10, 2: 1}
    ]
}

# กำหนด Metric สำหรับประเมินโมเดล
scoring = {
    "macro_f1": "f1_macro",
    "balanced_accuracy": "balanced_accuracy",
    "neg_log_loss": "neg_log_loss"
}

# สร้าง Grid Search
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring=scoring,
    refit="macro_f1",
    cv=cv_splits,
    n_jobs=-1,
    verbose=2,
    return_train_score=True,
    error_score="raise"
)

print("จำนวน Hyperparameter Combinations:",
      len(grid_search.param_grid["model__C"])
      * len(grid_search.param_grid["model__l1_ratio"])
      * len(param_grid["model__class_weight"]))

print("จำนวน Model Fits:", 75 * len(cv_splits))


จำนวน Hyperparameter Combinations: 75
จำนวน Model Fits: 375


In [47]:

# แยก Features และ Target สำหรับ Grid Search
X_train_cv = train_cv_df[features]
y_train_cv = train_cv_df[target_col]

# เริ่มค้นหา Hyperparameters
grid_search.fit(X_train_cv, y_train_cv)

print("Grid Search เสร็จสิ้น")
print("Best Parameters:")
print(grid_search.best_params_)

print("\nBest Mean CV Macro F1:")
print(grid_search.best_score_)


Fitting 5 folds for each of 75 candidates, totalling 375 fits
Grid Search เสร็จสิ้น
Best Parameters:
{'model__C': 10, 'model__class_weight': {0: 10, 1: 10, 2: 1}, 'model__l1_ratio': 0}

Best Mean CV Macro F1:
0.3936269750915103


In [48]:

# แปลงผล Grid Search เป็น DataFrame
cv_results = pd.DataFrame(grid_search.cv_results_)

# เลือกคอลัมน์ที่ต้องการตรวจสอบ
result_columns = [
    "param_model__C",
    "param_model__l1_ratio",
    "param_model__class_weight",
    "mean_test_macro_f1",
    "std_test_macro_f1",
    "mean_test_balanced_accuracy",
    "mean_test_neg_log_loss",
    "mean_train_macro_f1"
]

# เรียงตาม Mean CV Macro F1
top_results = (
    cv_results[result_columns]
    .sort_values(
        "mean_test_macro_f1",
        ascending=False
    )
    .head(15)
)

print(top_results.to_string(index=False))


 param_model__C  param_model__l1_ratio param_model__class_weight  mean_test_macro_f1  std_test_macro_f1  mean_test_balanced_accuracy  mean_test_neg_log_loss  mean_train_macro_f1
          10.00                    0.0      {0: 10, 1: 10, 2: 1}            0.393627           0.008106                     0.399983               -0.749305             0.398295
         100.00                    0.5      {0: 10, 1: 10, 2: 1}            0.393627           0.008106                     0.399983               -0.749305             0.398295
          10.00                    1.0      {0: 10, 1: 10, 2: 1}            0.393627           0.008106                     0.399983               -0.749305             0.398295
         100.00                    1.0      {0: 10, 1: 10, 2: 1}            0.393627           0.008106                     0.399983               -0.749305             0.398295
          10.00                    0.5      {0: 10, 1: 10, 2: 1}            0.393627           0.008106       

In [49]:
# ตรวจสอบผลการทดสอบแยกตาม Fold

best_idx = grid_search.best_index_

fold_scores = []

for fold in range(5):

    fold_scores.append({
        "Fold": fold + 1,
        "Train_Macro_F1": cv_results.loc[
            best_idx, f"split{fold}_train_macro_f1"
        ],
        "Val_Macro_F1": cv_results.loc[
            best_idx, f"split{fold}_test_macro_f1"
        ]
    })

fold_scores_df = pd.DataFrame(fold_scores)

print("Best Model: Macro F1 by Fold")
print(fold_scores_df.round(4).to_string(index=False))

print("\nMean Validation Macro F1:",
      round(fold_scores_df["Val_Macro_F1"].mean(), 6))

print("Std Validation Macro F1:",
      round(fold_scores_df["Val_Macro_F1"].std(ddof=0), 6))


Best Model: Macro F1 by Fold
 Fold  Train_Macro_F1  Val_Macro_F1
    1          0.3960        0.3983
    2          0.4004        0.3982
    3          0.3940        0.4024
    4          0.3979        0.3893
    5          0.4032        0.3798

Mean Validation Macro F1: 0.393627
Std Validation Macro F1: 0.008106


In [50]:



# เก็บผลการประเมินรายคลาสของแต่ละ Fold
class_metrics = []

for fold, (train_idx, val_idx) in enumerate(cv_splits, start=1):

    # แยกข้อมูลตาม Fold
    X_fold_train = X_train_cv.iloc[train_idx]
    y_fold_train = y_train_cv.iloc[train_idx]

    X_fold_val = X_train_cv.iloc[val_idx]
    y_fold_val = y_train_cv.iloc[val_idx]

    # สร้างโมเดลจาก Best Configuration
    fold_model = clone(grid_search.best_estimator_)

    # ฝึกโมเดลด้วย Training Portion ของ Fold
    fold_model.fit(X_fold_train, y_fold_train)

    # ทำนาย Validation Portion
    y_fold_pred = fold_model.predict(X_fold_val)

    # คำนวณผลการจำแนกแต่ละคลาส
    report = classification_report(
        y_fold_val,
        y_fold_pred,
        labels=[0, 1, 2],
        target_names=["Panic", "FOMO", "Normal"],
        output_dict=True,
        zero_division=0
    )

    # เก็บผลลัพธ์
    for class_name in ["Panic", "FOMO", "Normal"]:

        class_metrics.append({
            "Fold": fold,
            "Class": class_name,
            "Precision": report[class_name]["precision"],
            "Recall": report[class_name]["recall"],
            "F1": report[class_name]["f1-score"],
            "Support": report[class_name]["support"]
        })

# แสดงผลลัพธ์รายคลาส
class_metrics_df = pd.DataFrame(class_metrics)

print(class_metrics_df.round(4).to_string(index=False))


 Fold  Class  Precision  Recall     F1  Support
    1  Panic     0.1576  0.0936 0.1174    652.0
    1   FOMO     0.1325  0.2269 0.1673    780.0
    1 Normal     0.9202  0.9006 0.9103  13681.0
    2  Panic     0.1330  0.1402 0.1365    699.0
    2   FOMO     0.1287  0.1941 0.1548    819.0
    2 Normal     0.9188  0.8885 0.9034  13764.0
    3  Panic     0.1424  0.0790 0.1017    620.0
    3   FOMO     0.1614  0.2299 0.1897    957.0
    3 Normal     0.9201  0.9117 0.9159  14367.0
    4  Panic     0.1521  0.0870 0.1106    759.0
    4   FOMO     0.1118  0.1893 0.1405    708.0
    4 Normal     0.9220  0.9116 0.9168  14710.0
    5  Panic     0.1187  0.0906 0.1027    806.0
    5   FOMO     0.1056  0.1448 0.1221    753.0
    5 Normal     0.9173  0.9120 0.9147  15198.0


In [51]:

# เก็บผลการทำนายแยกตาม Fold
oof_results = []

for fold, (train_idx, val_idx) in enumerate(cv_splits, start=1):

    # แยกข้อมูลตาม Fold
    X_fold_train = X_train_cv.iloc[train_idx]
    y_fold_train = y_train_cv.iloc[train_idx]

    X_fold_val = X_train_cv.iloc[val_idx]
    y_fold_val = y_train_cv.iloc[val_idx]

    # สร้างและฝึกโมเดล
    fold_model = clone(grid_search.best_estimator_)
    fold_model.fit(X_fold_train, y_fold_train)

    # ทำนายคลาสและ Probability
    y_fold_pred = fold_model.predict(X_fold_val)
    y_fold_proba = fold_model.predict_proba(X_fold_val)

    # เก็บผลการทำนายพร้อมตัวระบุหุ้น
    fold_result = pd.DataFrame({
        "Fold": fold,
        "Target_Date": train_cv_df.iloc[val_idx]["Target_Date"].values,
        "Symbol": train_cv_df.iloc[val_idx]["Symbol"].values,
        "Actual": y_fold_val.values,
        "Predicted": y_fold_pred,
        "Prob_Panic": y_fold_proba[:, 0],
        "Prob_FOMO": y_fold_proba[:, 1],
        "Prob_Normal": y_fold_proba[:, 2]
    })

    oof_results.append(fold_result)

# รวมผลการทำนายทุก Fold
oof_df = pd.concat(oof_results, ignore_index=True)

# ตรวจสอบตัวอย่างซ้ำด้วย Target Date และ Symbol
duplicate_count = oof_df.duplicated(
    subset=["Target_Date", "Symbol"]
).sum()

print("Total OOF Predictions:", len(oof_df))
print("Duplicate Date-Symbol:", duplicate_count)

# ตรวจสอบจำนวนตัวอย่างต่อ Fold
print("\nSamples by Fold:")
print(oof_df.groupby("Fold").size())

# ตรวจสอบจำนวนหุ้นและวันที่
print("\nUnique Symbols:", oof_df["Symbol"].nunique())
print("Unique Target Dates:", oof_df["Target_Date"].nunique())

# ตรวจสอบว่าแต่ละตัวอย่างอยู่ใน Fold เดียวเท่านั้น
assert duplicate_count == 0

print("\nตรวจสอบ OOF Identity ผ่าน")


Total OOF Predictions: 79273
Duplicate Date-Symbol: 0

Samples by Fold:
Fold
1    15113
2    15282
3    15944
4    16177
5    16757
dtype: int64

Unique Symbols: 69
Unique Target Dates: 1770

ตรวจสอบ OOF Identity ผ่าน


In [52]:

# ตรวจสอบการกระจาย Probability ของแต่ละคลาส
probability_summary = oof_df[
    ["Prob_Panic", "Prob_FOMO", "Prob_Normal"]
].describe(
    percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
).T

print("Probability Distribution:")
print(probability_summary.round(4).to_string())

# ตรวจสอบ Probability สูงสุดของแต่ละคลาส
for col in ["Prob_Panic", "Prob_FOMO", "Prob_Normal"]:

    print(f"\n{col}")
    print("Top 10 Probability Values:")
    print(
        oof_df[col]
        .nlargest(10)
        .round(4)
        .to_list()
    )

# ตรวจสอบผลรวม Probability
prob_sum = oof_df[
    ["Prob_Panic", "Prob_FOMO", "Prob_Normal"]
].sum(axis=1)

print("\nProbability Sum Check:")
print("Minimum:", round(prob_sum.min(), 8))
print("Maximum:", round(prob_sum.max(), 8))

assert np.allclose(prob_sum, 1.0)


Probability Distribution:
               count    mean     std     min     50%     75%     90%     95%     99%     max
Prob_Panic   79273.0  0.2095  0.0748  0.0001  0.1954  0.2353  0.2878  0.3366  0.5008  0.9989
Prob_FOMO    79273.0  0.2778  0.0785  0.0009  0.2658  0.3130  0.3700  0.4181  0.5535  0.9999
Prob_Normal  79273.0  0.5126  0.1137  0.0000  0.5308  0.5911  0.6363  0.6604  0.7033  0.8128

Prob_Panic
Top 10 Probability Values:
[0.9989, 0.9982, 0.9963, 0.9952, 0.995, 0.9942, 0.9896, 0.9885, 0.9848, 0.9841]

Prob_FOMO
Top 10 Probability Values:
[0.9999, 0.9654, 0.952, 0.9175, 0.9144, 0.9098, 0.9081, 0.8976, 0.8874, 0.883]

Prob_Normal
Top 10 Probability Values:
[0.8128, 0.8036, 0.7994, 0.7971, 0.7952, 0.7901, 0.7864, 0.7847, 0.7826, 0.7819]

Probability Sum Check:
Minimum: 1.0
Maximum: 1.0


In [53]:



# กำหนดช่วง Threshold สำหรับค้นหา
threshold_values = np.round(
    np.arange(0.10, 0.551, 0.01),
    2
)

# เตรียม Probability และ Target
p_panic = oof_df["Prob_Panic"].to_numpy()
p_fomo = oof_df["Prob_FOMO"].to_numpy()
y_true = oof_df["Actual"].to_numpy()

# คำนวณผล Baseline ด้วย Argmax
y_argmax = np.argmax(
    oof_df[["Prob_Panic", "Prob_FOMO", "Prob_Normal"]].to_numpy(),
    axis=1
)

baseline_macro_f1 = f1_score(
    y_true,
    y_argmax,
    labels=[0, 1, 2],
    average="macro"
)

# เก็บผลการค้นหา Threshold
threshold_results = []

# ทดลอง Threshold ของ Panic และ FOMO ทุกคู่
for t_panic in threshold_values:

    for t_fomo in threshold_values:

        # ตรวจสอบว่า Probability ผ่าน Threshold หรือไม่
        panic_pass = p_panic >= t_panic
        fomo_pass = p_fomo >= t_fomo

        # กำหนดผลการทำนายเริ่มต้นเป็น Normal
        y_pred = np.full(len(y_true), 2, dtype=int)

        # กรณีผ่านเฉพาะ Panic
        y_pred[panic_pass & ~fomo_pass] = 0

        # กรณีผ่านเฉพาะ FOMO
        y_pred[fomo_pass & ~panic_pass] = 1

        # กรณีผ่านทั้งสองคลาส เลือก Probability ที่สูงกว่า
        both_pass = panic_pass & fomo_pass

        y_pred[both_pass] = np.where(
            p_panic[both_pass] >= p_fomo[both_pass],
            0,
            1
        )

        # คำนวณผลการประเมิน
        threshold_results.append({
            "Threshold_Panic": t_panic,
            "Threshold_FOMO": t_fomo,
            "Macro_F1": f1_score(
                y_true,
                y_pred,
                labels=[0, 1, 2],
                average="macro"
            ),
            "Panic_Precision": precision_score(
                y_true, y_pred, labels=[0, 1, 2],
                average=None, zero_division=0
            )[0],
            "Panic_Recall": recall_score(
                y_true, y_pred, labels=[0, 1, 2],
                average=None, zero_division=0
            )[0],
            "FOMO_Precision": precision_score(
                y_true, y_pred, labels=[0, 1, 2],
                average=None, zero_division=0
            )[1],
            "FOMO_Recall": recall_score(
                y_true, y_pred, labels=[0, 1, 2],
                average=None, zero_division=0
            )[1]
        })

# รวมผลการค้นหา
threshold_results_df = pd.DataFrame(threshold_results)

# เรียงตาม Macro F1
threshold_results_df = threshold_results_df.sort_values(
    ["Macro_F1", "Panic_Precision", "FOMO_Precision"],
    ascending=False
).reset_index(drop=True)

# แสดงผล
print("Argmax Baseline Macro F1:",
      round(baseline_macro_f1, 6))

print("\nTop 15 Threshold Combinations:")
print(
    threshold_results_df.head(15)
    .round(4)
    .to_string(index=False)
)

print("\nBest Threshold Configuration:")
print(threshold_results_df.iloc[0].to_string())


Argmax Baseline Macro F1: 0.394655

Top 15 Threshold Combinations:
 Threshold_Panic  Threshold_FOMO  Macro_F1  Panic_Precision  Panic_Recall  FOMO_Precision  FOMO_Recall
            0.31            0.39    0.4032           0.1251        0.1751          0.1384       0.1964
            0.32            0.39    0.4031           0.1292        0.1581          0.1384       0.1964
            0.31            0.40    0.4026           0.1252        0.1779          0.1410       0.1745
            0.32            0.40    0.4026           0.1296        0.1606          0.1410       0.1745
            0.33            0.39    0.4024           0.1325        0.1428          0.1384       0.1964
            0.29            0.39    0.4024           0.1177        0.2212          0.1384       0.1964
            0.30            0.39    0.4023           0.1198        0.1923          0.1384       0.1964
            0.31            0.41    0.4019           0.1257        0.1810          0.1443       0.1551
      

In [54]:

# กำหนด Threshold จากผล OOF
best_t_panic = 0.31
best_t_fomo = 0.39

# ทำนายความน่าจะเป็นบน Validation
val_proba = grid_search.best_estimator_.predict_proba(X_val)

# แยก Probability ของแต่ละคลาส
p_panic = val_proba[:, 0]
p_fomo = val_proba[:, 1]

# สร้างผลทำนายด้วย Threshold
panic_pass = p_panic >= best_t_panic
fomo_pass = p_fomo >= best_t_fomo

y_val_pred_threshold = np.full(len(y_val), 2, dtype=int)

# กรณีผ่านเฉพาะ Panic
y_val_pred_threshold[panic_pass & ~fomo_pass] = 0

# กรณีผ่านเฉพาะ FOMO
y_val_pred_threshold[fomo_pass & ~panic_pass] = 1

# กรณีผ่านทั้งสองคลาส
both_pass = panic_pass & fomo_pass

y_val_pred_threshold[both_pass] = np.where(
    p_panic[both_pass] >= p_fomo[both_pass],
    0,
    1
)

# ประเมินผล
print("Validation Macro F1:",
      round(f1_score(y_val, y_val_pred_threshold, average="macro"), 6))

print("Validation Accuracy:",
      round(accuracy_score(y_val, y_val_pred_threshold), 6))

print("\nClassification Report:")
print(classification_report(
    y_val,
    y_val_pred_threshold,
    labels=[0, 1, 2],
    target_names=["Panic", "FOMO", "Normal"],
    zero_division=0
))

print("\nConfusion Matrix:")
print(confusion_matrix(
    y_val,
    y_val_pred_threshold,
    labels=[0, 1, 2]
))

Validation Macro F1: 0.403221
Validation Accuracy: 0.805903

Classification Report:
              precision    recall  f1-score   support

       Panic       0.13      0.26      0.17       544
        FOMO       0.14      0.15      0.14       617
      Normal       0.92      0.87      0.90     10663

    accuracy                           0.81     11824
   macro avg       0.40      0.42      0.40     11824
weighted avg       0.84      0.81      0.82     11824


Confusion Matrix:
[[ 139   53  352]
 [  83   90  444]
 [ 877  486 9300]]


In [55]:
# สร้างผลทำนายแบบ Argmax
y_val_pred_argmax = np.argmax(val_proba, axis=1)

# สร้างตารางเปรียบเทียบ
comparison_results = []

for method, y_pred in [
    ("Argmax", y_val_pred_argmax),
    ("Threshold", y_val_pred_threshold)
]:
    report = classification_report(
        y_val,
        y_pred,
        labels=[0, 1, 2],
        output_dict=True,
        zero_division=0
    )

    comparison_results.append({
        "Method": method,
        "Accuracy": accuracy_score(y_val, y_pred),
        "Macro_F1": f1_score(
            y_val, y_pred,
            labels=[0, 1, 2],
            average="macro"
        ),
        "Panic_F1": report["0"]["f1-score"],
        "Panic_Recall": report["0"]["recall"],
        "FOMO_F1": report["1"]["f1-score"],
        "FOMO_Recall": report["1"]["recall"],
        "Normal_F1": report["2"]["f1-score"],
        "Normal_Recall": report["2"]["recall"]
    })

comparison_df = pd.DataFrame(comparison_results)

# แสดงผลการเปรียบเทียบ
print(comparison_df.round(4).to_string(index=False))

# ตรวจสอบสัดส่วนการทำนายแต่ละคลาส
prediction_distribution = pd.DataFrame({
    "Argmax": pd.Series(y_val_pred_argmax).value_counts(normalize=True),
    "Threshold": pd.Series(y_val_pred_threshold).value_counts(normalize=True)
}).fillna(0)

prediction_distribution.index = [
    ["Panic", "FOMO", "Normal"][i]
    for i in prediction_distribution.index
]

print("\nPrediction Distribution:")
print((prediction_distribution * 100).round(2))

   Method  Accuracy  Macro_F1  Panic_F1  Panic_Recall  FOMO_F1  FOMO_Recall  Normal_F1  Normal_Recall
   Argmax    0.8409    0.4159    0.1668        0.1599   0.1644       0.1783     0.9165         0.9140
Threshold    0.8059    0.4032    0.1692        0.2555   0.1445       0.1459     0.8960         0.8722

Prediction Distribution:
        Argmax  Threshold
Panic     4.22       9.29
FOMO      6.10       5.32
Normal   89.68      85.39


In [56]:

# เตรียมข้อมูลให้ตรงกับลำดับของ Cross-Validation
X_cv = train_cv_df[features].copy()
y_cv = train_cv_df[target_col].astype(int).copy()

# กำหนดน้ำหนักของแต่ละคลาส
panic_weights = [2, 5, 10, 15, 20]
fomo_weights = [2, 5, 10, 15, 20]

# สร้างชุดน้ำหนักทุกคู่
class_weight_grid = [
    {0: panic_weight, 1: fomo_weight, 2: 1}
    for panic_weight in panic_weights
    for fomo_weight in fomo_weights
]

# สร้าง Pipeline
weight_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        C=10,
        solver="saga",
        l1_ratio=0,
        class_weight=None,
        max_iter=5000,
        random_state=42
    ))
])

# กำหนดค่าที่ต้องการค้นหา
weight_param_grid = {
    "model__class_weight": class_weight_grid
}

# กำหนด Metric สำหรับการประเมิน
scoring = {
    "macro_f1": "f1_macro",
    "balanced_accuracy": "balanced_accuracy",
    "neg_log_loss": "neg_log_loss"
}

# ตั้งค่า Grid Search
grid_weight = GridSearchCV(
    estimator=weight_pipeline,
    param_grid=weight_param_grid,
    scoring=scoring,
    refit="macro_f1",
    cv=cv_splits,
    n_jobs=-1,
    verbose=2,
    return_train_score=True,
    error_score="raise"
)

# เริ่มค้นหา Hyperparameters
grid_weight.fit(X_cv, y_cv)

# เก็บโมเดลที่ดีที่สุด
advanced_weight_model = grid_weight.best_estimator_

# แสดงผลการค้นหา
advanced_weight_results = pd.DataFrame(grid_weight.cv_results_)

advanced_weight_results = advanced_weight_results.sort_values(
    "mean_test_macro_f1",
    ascending=False
).reset_index(drop=True)

# เลือกผลลัพธ์ที่ต้องการแสดง
weight_display_cols = [
    "param_model__class_weight",
    "mean_test_macro_f1",
    "std_test_macro_f1",
    "mean_test_balanced_accuracy",
    "mean_test_neg_log_loss",
    "mean_train_macro_f1"
]

print("Best Parameters:")
print(grid_weight.best_params_)

print("\nBest Mean CV Macro F1:")
print(round(grid_weight.best_score_, 6))

print("\nTop 15 Results:")
print(
    advanced_weight_results[weight_display_cols]
    .head(15)
    .round(4)
    .to_string(index=False)
)


Fitting 5 folds for each of 25 candidates, totalling 125 fits
Best Parameters:
{'model__class_weight': {0: 15, 1: 10, 2: 1}}

Best Mean CV Macro F1:
0.397738

Top 15 Results:
param_model__class_weight  mean_test_macro_f1  std_test_macro_f1  mean_test_balanced_accuracy  mean_test_neg_log_loss  mean_train_macro_f1
     {0: 15, 1: 10, 2: 1}              0.3977             0.0125                       0.4258                 -0.8300               0.4011
     {0: 10, 1: 10, 2: 1}              0.3936             0.0081                       0.4000                 -0.7493               0.3983
      {0: 10, 1: 5, 2: 1}              0.3755             0.0100                       0.3774                 -0.6343               0.3743
     {0: 15, 1: 15, 2: 1}              0.3715             0.0103                       0.4379                 -0.9268               0.3780
      {0: 5, 1: 10, 2: 1}              0.3694             0.0106                       0.3855                 -0.6684             

In [57]:


# สร้างโมเดล EasyEnsemble สำหรับข้อมูล 3 คลาส
class EasyEnsembleSoftmax(ClassifierMixin, BaseEstimator):

    def __init__(
        self,
        n_estimators=10,
        normal_ratio=1.0,
        C=10.0,
        max_iter=5000,
        random_state=42
    ):
        self.n_estimators = n_estimators
        self.normal_ratio = normal_ratio
        self.C = C
        self.max_iter = max_iter
        self.random_state = random_state

    def fit(self, X, y):

        # ตรวจสอบข้อมูลฝึก
        X, y = check_X_y(X, y)

        self.classes_ = np.unique(y)

        # ตรวจสอบว่ามีครบทั้งสามคลาส
        if not np.array_equal(
            self.classes_,
            np.array([0, 1, 2])
        ):
            raise ValueError(
                "Training Fold ต้องมี Target ครบทั้ง 3 คลาส"
            )

        # แยกตำแหน่งของแต่ละคลาส
        panic_idx = np.flatnonzero(y == 0)
        fomo_idx = np.flatnonzero(y == 1)
        normal_idx = np.flatnonzero(y == 2)

        # เก็บตัวอย่าง Panic และ FOMO ทั้งหมด
        minority_idx = np.concatenate([
            panic_idx,
            fomo_idx
        ])

        # คำนวณจำนวน Normal ที่จะสุ่มในแต่ละโมเดล
        n_normal = int(np.ceil(
            self.normal_ratio * len(minority_idx)
        ))

        n_normal = min(
            n_normal,
            len(normal_idx)
        )

        if n_normal < 1:
            raise ValueError("จำนวนตัวอย่าง Normal ไม่เพียงพอ")

        # สร้างตัวสุ่มสำหรับแต่ละชุด
        rng = np.random.default_rng(self.random_state)

        self.estimators_ = []
        self.n_features_in_ = X.shape[1]

        # ฝึก Logistic Regression หลายโมเดล
        for i in range(self.n_estimators):

            # สุ่มตัวอย่าง Normal โดยไม่แทนที่
            sampled_normal = rng.choice(
                normal_idx,
                size=n_normal,
                replace=False
            )

            # รวม Minority และ Normal ที่สุ่มได้
            selected_idx = np.concatenate([
                minority_idx,
                sampled_normal
            ])

            # สุ่มลำดับตัวอย่างก่อนฝึก
            selected_idx = rng.permutation(selected_idx)

            # สร้างและฝึก Logistic Regression
            model = LogisticRegression(
                C=self.C,
                solver="saga",
                l1_ratio=0,
                class_weight=None,
                max_iter=self.max_iter,
                random_state=self.random_state + i
            )

            model.fit(
                X[selected_idx],
                y[selected_idx]
            )

            self.estimators_.append(model)

        return self

    def predict_proba(self, X):

        # ตรวจสอบว่าโมเดลผ่านการฝึกแล้ว
        check_is_fitted(self, "estimators_")

        X = check_array(X)

        # เฉลี่ย Probability จากทุกโมเดล
        all_probabilities = np.stack([
            model.predict_proba(X)
            for model in self.estimators_
        ])

        return np.mean(all_probabilities, axis=0)

    def predict(self, X):

        # เลือกคลาสที่มี Probability สูงที่สุด
        probabilities = self.predict_proba(X)

        return self.classes_[
            np.argmax(probabilities, axis=1)
        ]


# เตรียมข้อมูลให้ตรงกับลำดับของ Cross-Validation
X_cv = train_cv_df[features].copy()
y_cv = train_cv_df[target_col].astype(int).copy()

# สร้าง Pipeline
easy_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", EasyEnsembleSoftmax(
        C=10,
        max_iter=5000,
        random_state=42
    ))
])

# กำหนดค่าที่ต้องการค้นหา
easy_param_grid = {
    "model__n_estimators": [10, 20],
    "model__normal_ratio": [1.0, 2.0, 4.0]
}

# กำหนด Metric สำหรับการประเมิน
scoring = {
    "macro_f1": "f1_macro",
    "balanced_accuracy": "balanced_accuracy",
    "neg_log_loss": "neg_log_loss"
}

# ตั้งค่า Grid Search
grid_easy = GridSearchCV(
    estimator=easy_pipeline,
    param_grid=easy_param_grid,
    scoring=scoring,
    refit="macro_f1",
    cv=cv_splits,
    n_jobs=-1,
    verbose=2,
    return_train_score=True,
    error_score="raise"
)

# เริ่มค้นหา Hyperparameters
grid_easy.fit(X_cv, y_cv)

# เก็บโมเดลที่ดีที่สุด
easyensemble_model = grid_easy.best_estimator_

# แสดงผลการค้นหา
easyensemble_results = pd.DataFrame(grid_easy.cv_results_)

easyensemble_results = easyensemble_results.sort_values(
    "mean_test_macro_f1",
    ascending=False
).reset_index(drop=True)

# เลือกผลลัพธ์ที่ต้องการแสดง
easy_display_cols = [
    "param_model__n_estimators",
    "param_model__normal_ratio",
    "mean_test_macro_f1",
    "std_test_macro_f1",
    "mean_test_balanced_accuracy",
    "mean_test_neg_log_loss",
    "mean_train_macro_f1"
]

print("Best Parameters:")
print(grid_easy.best_params_)

print("\nBest Mean CV Macro F1:")
print(round(grid_easy.best_score_, 6))

print("\nTop Results:")
print(
    easyensemble_results[easy_display_cols]
    .round(4)
    .to_string(index=False)
)


Fitting 5 folds for each of 6 candidates, totalling 30 fits
Best Parameters:
{'model__n_estimators': 10, 'model__normal_ratio': 1.0}

Best Mean CV Macro F1:
0.392461

Top Results:
 param_model__n_estimators  param_model__normal_ratio  mean_test_macro_f1  std_test_macro_f1  mean_test_balanced_accuracy  mean_test_neg_log_loss  mean_train_macro_f1
                        10                        1.0              0.3925             0.0084                       0.3945                 -0.7250               0.3973
                        20                        1.0              0.3922             0.0084                       0.3942                 -0.7251               0.3975
                        20                        2.0              0.3625             0.0095                       0.3574                 -0.5184               0.3648
                        10                        2.0              0.3624             0.0094                       0.3573                 -0.5184       

In [58]:
# กำหนดจำนวนตัวอย่างเป้าหมายของแต่ละคลาส
def smote_target_counts(y, ratio):

    # นับจำนวนตัวอย่างของแต่ละคลาส
    class_counts = np.bincount(
        np.asarray(y, dtype=int),
        minlength=3
    )

    # ใช้จำนวน Normal เป็นฐานในการคำนวณ
    normal_count = class_counts[2]

    # คำนวณจำนวนตัวอย่างเป้าหมายของ Panic และ FOMO
    panic_target = max(
        class_counts[0],
        int(np.ceil(ratio * normal_count))
    )

    fomo_target = max(
        class_counts[1],
        int(np.ceil(ratio * normal_count))
    )

    return {
        0: panic_target,
        1: fomo_target
    }


# เตรียมข้อมูลให้ตรงกับลำดับของ Cross-Validation
X_cv = train_cv_df[features].copy()
y_cv = train_cv_df[target_col].astype(int).copy()

# กำหนดสัดส่วนการเพิ่มตัวอย่าง
smote_ratios = [0.10, 0.25, 0.50]

# สร้าง Sampling Strategy สำหรับแต่ละสัดส่วน
smote_strategies = [
    partial(smote_target_counts, ratio=ratio)
    for ratio in smote_ratios
]

# สร้าง Pipeline
smote_pipeline = ImbPipeline([
    ("scaler", StandardScaler()),
    ("smote", SMOTE(
        random_state=42
    )),
    ("model", LogisticRegression(
        C=10,
        solver="saga",
        l1_ratio=0,
        class_weight=None,
        max_iter=5000,
        random_state=42
    ))
])

# กำหนดค่าที่ต้องการค้นหา
smote_param_grid = {
    "smote__sampling_strategy": smote_strategies,
    "smote__k_neighbors": [3, 5]
}

# กำหนด Metric สำหรับการประเมิน
scoring = {
    "macro_f1": "f1_macro",
    "balanced_accuracy": "balanced_accuracy",
    "neg_log_loss": "neg_log_loss"
}

# ตั้งค่า Grid Search
grid_smote = GridSearchCV(
    estimator=smote_pipeline,
    param_grid=smote_param_grid,
    scoring=scoring,
    refit="macro_f1",
    cv=cv_splits,
    n_jobs=-1,
    verbose=2,
    return_train_score=True,
    error_score="raise"
)

# เริ่มค้นหา Hyperparameters
grid_smote.fit(X_cv, y_cv)

# เก็บโมเดลที่ดีที่สุด
smote_model = grid_smote.best_estimator_

# แสดงผลการค้นหา
smote_results = pd.DataFrame(grid_smote.cv_results_)

# แปลง Sampling Strategy กลับเป็นสัดส่วนที่อ่านง่าย
smote_results["SMOTE_Ratio"] = (
    smote_results["param_smote__sampling_strategy"]
    .apply(lambda strategy: strategy.keywords["ratio"])
)

smote_results = smote_results.sort_values(
    "mean_test_macro_f1",
    ascending=False
).reset_index(drop=True)

# เลือกผลลัพธ์ที่ต้องการแสดง
smote_display_cols = [
    "SMOTE_Ratio",
    "param_smote__k_neighbors",
    "mean_test_macro_f1",
    "std_test_macro_f1",
    "mean_test_balanced_accuracy",
    "mean_test_neg_log_loss",
    "mean_train_macro_f1"
]

print("Best SMOTE Ratio:")
print(
    grid_smote.best_params_[
        "smote__sampling_strategy"
    ].keywords["ratio"]
)

print("\nBest K Neighbors:")
print(grid_smote.best_params_["smote__k_neighbors"])

print("\nBest Mean CV Macro F1:")
print(round(grid_smote.best_score_, 6))

print("\nTop Results:")
print(
    smote_results[smote_display_cols]
    .round(4)
    .to_string(index=False)
)


Fitting 5 folds for each of 6 candidates, totalling 30 fits
Best SMOTE Ratio:
0.5

Best K Neighbors:
3

Best Mean CV Macro F1:
0.397964

Top Results:
 SMOTE_Ratio  param_smote__k_neighbors  mean_test_macro_f1  std_test_macro_f1  mean_test_balanced_accuracy  mean_test_neg_log_loss  mean_train_macro_f1
        0.50                         3              0.3980             0.0108                       0.3983                 -0.7267               0.4014
        0.50                         5              0.3979             0.0101                       0.3989                 -0.7262               0.4015
        0.25                         3              0.3635             0.0098                       0.3582                 -0.5198               0.3683
        0.25                         5              0.3625             0.0098                       0.3575                 -0.5193               0.3674
        0.10                         3              0.3335             0.0045             

In [59]:

# กำหนดโมเดลที่ต้องการเปรียบเทียบ
candidate_models = {
    "Original": grid_search.best_estimator_,
    "Advanced_Weight": advanced_weight_model,
    "EasyEnsemble": easyensemble_model,
    "SMOTE": smote_model
}

# เก็บผลการประเมิน
validation_results = []
validation_predictions = {}
validation_probabilities = {}

# ประเมินโมเดลแต่ละตัว
for model_name, model in candidate_models.items():

    # ทำนายคลาสและความน่าจะเป็น
    y_pred = model.predict(X_val)
    y_proba = model.predict_proba(X_val)

    # เก็บผลการทำนาย
    validation_predictions[model_name] = y_pred
    validation_probabilities[model_name] = y_proba

    # คำนวณผลการประเมิน
    report = classification_report(
        y_val,
        y_pred,
        labels=[0, 1, 2],
        output_dict=True,
        zero_division=0
    )

    validation_results.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_val, y_pred),
        "Macro_F1": f1_score(
            y_val, y_pred,
            labels=[0, 1, 2],
            average="macro"
        ),
        "Balanced_Accuracy": balanced_accuracy_score(
            y_val, y_pred
        ),
        "Log_Loss": log_loss(
            y_val,
            y_proba,
            labels=[0, 1, 2]
        ),
        "Panic_Precision": report["0"]["precision"],
        "Panic_Recall": report["0"]["recall"],
        "Panic_F1": report["0"]["f1-score"],
        "FOMO_Precision": report["1"]["precision"],
        "FOMO_Recall": report["1"]["recall"],
        "FOMO_F1": report["1"]["f1-score"],
        "Normal_F1": report["2"]["f1-score"],
        "Normal_Recall": report["2"]["recall"]
    })

# รวมผลการประเมินเป็นตาราง
validation_results_df = pd.DataFrame(validation_results)

validation_results_df = validation_results_df.sort_values(
    "Macro_F1",
    ascending=False
).reset_index(drop=True)

# แสดงผลการเปรียบเทียบ
print("Validation Results:")
print(
    validation_results_df
    .round(4)
    .to_string(index=False)
)

# แสดง Confusion Matrix ของแต่ละโมเดล
for model_name, y_pred in validation_predictions.items():

    print(f"\n{'=' * 50}")
    print(f"Model: {model_name}")
    print("Confusion Matrix:")

    print(confusion_matrix(
        y_val,
        y_pred,
        labels=[0, 1, 2]
    ))

# เปรียบเทียบสัดส่วนการทำนายแต่ละคลาส
distribution_results = []

for model_name, y_pred in validation_predictions.items():

    counts = pd.Series(y_pred).value_counts(normalize=True)

    distribution_results.append({
        "Model": model_name,
        "Panic_Pct": counts.get(0, 0) * 100,
        "FOMO_Pct": counts.get(1, 0) * 100,
        "Normal_Pct": counts.get(2, 0) * 100
    })

distribution_df = pd.DataFrame(distribution_results)

print("\nPrediction Distribution (%):")
print(distribution_df.round(2).to_string(index=False))


Validation Results:
          Model  Accuracy  Macro_F1  Balanced_Accuracy  Log_Loss  Panic_Precision  Panic_Recall  Panic_F1  FOMO_Precision  FOMO_Recall  FOMO_F1  Normal_F1  Normal_Recall
       Original    0.8409    0.4159             0.4174    0.7501           0.1743        0.1599    0.1668          0.1526       0.1783   0.1644     0.9165         0.9140
   EasyEnsemble    0.8490    0.4133             0.4103    0.7294           0.1804        0.1489    0.1631          0.1537       0.1572   0.1554     0.9213         0.9248
          SMOTE    0.8471    0.4081             0.4088    0.7320           0.1633        0.1930    0.1769          0.1495       0.1102   0.1269     0.9204         0.9231
Advanced_Weight    0.7800    0.3887             0.4312    0.8381           0.1142        0.3585    0.1733          0.1368       0.0940   0.1114     0.8813         0.8412

Model: Original
Confusion Matrix:
[[  87   65  392]
 [  41  110  466]
 [ 371  546 9746]]

Model: Advanced_Weight
Confusion Matrix

In [60]:

# ตรวจสอบจำนวนตัวอย่างแต่ละคลาสในแต่ละ Period ของทุก Fold

period_smote_audit = []

for fold_id, (train_idx, val_idx) in enumerate(cv_splits, start=1):

    fold_train = train_cv_df.iloc[train_idx]

    period_counts = (
        fold_train
        .groupby(["Period", target_col])
        .size()
        .unstack(fill_value=0)
        .reindex(columns=[0, 1, 2], fill_value=0)
    )

    for period, row in period_counts.iterrows():

        period_smote_audit.append({
            "Fold": fold_id,
            "Period": period,
            "Panic": int(row[0]),
            "FOMO": int(row[1]),
            "Normal": int(row[2]),
            "Min_Minority": int(min(row[0], row[1])),
            "Panic_K3": row[0] >= 4,
            "FOMO_K3": row[1] >= 4,
            "Panic_K5": row[0] >= 6,
            "FOMO_K5": row[1] >= 6
        })

period_smote_audit_df = pd.DataFrame(period_smote_audit)

print("จำนวนตัวอย่างในแต่ละ Period แยกตาม Fold")
print(period_smote_audit_df.to_string(index=False))

print("\nสรุปความพร้อมสำหรับ SMOTE")

print(
    period_smote_audit_df
    .groupby("Fold")
    .agg(
        Periods=("Period", "count"),
        Min_Panic=("Panic", "min"),
        Min_FOMO=("FOMO", "min"),
        Periods_Ready_K3=(
            "Min_Minority",
            lambda x: (x >= 4).sum()
        ),
        Periods_Ready_K5=(
            "Min_Minority",
            lambda x: (x >= 6).sum()
        )
    )
    .to_string()
)


จำนวนตัวอย่างในแต่ละ Period แยกตาม Fold
 Fold  Period  Panic  FOMO  Normal  Min_Minority  Panic_K3  FOMO_K3  Panic_K5  FOMO_K5
    1 2016_H1    102   139    2321           102      True     True      True     True
    1 2016_H2    176   283    4625           176      True     True      True     True
    1 2017_H1    214   282    4648           214      True     True      True     True
    1 2017_H2     86   156    1900            86      True     True      True     True
    2 2016_H1    102   139    2321           102      True     True      True     True
    2 2016_H2    176   283    4625           176      True     True      True     True
    2 2017_H1    214   282    4648           214      True     True      True     True
    2 2017_H2    186   327    4611           186      True     True      True     True
    2 2018_H1    284   232    4730           232      True     True      True     True
    2 2018_H2    207   267    4815           207      True     True      True     True
   

## สร้าง Period-aware SMOTE และทำ Grid Search

In [81]:
RANDOM_STATE = 42

In [82]:

# กำหนด Features และ Target
features = [
    "daily_return",
    "momentum_20d",
    "vol_z",
    "VolRank",
    "CorrRank",
    "LiquidityRank"
]

target_col = "target_aligned"

# เรียงข้อมูลให้ตรงกับลำดับวันที่และหุ้น
train_cv_df = (
    train_df
    .sort_values(["Target_Date", "Symbol"])
    .reset_index(drop=True)
    .copy()
)

# ตรวจสอบคอลัมน์ที่จำเป็น
required_cols = [
    "Target_Date",
    "Symbol",
    "Period",
    target_col
] + features

missing_cols = [
    col for col in required_cols
    if col not in train_cv_df.columns
]

assert len(missing_cols) == 0, f"Missing columns: {missing_cols}"

# ตรวจสอบข้อมูลที่ขาดหาย
assert train_cv_df[required_cols].isna().sum().sum() == 0

# แปลง Target เป็น Integer
train_cv_df[target_col] = train_cv_df[target_col].astype(int)

# ตรวจสอบ Target Classes
assert set(train_cv_df[target_col].unique()) == {0, 1, 2}

# สร้างรหัสประจำ Period
period_categories = sorted(train_cv_df["Period"].unique())

period_mapping = {
    period: index
    for index, period in enumerate(period_categories)
}

train_cv_df["PeriodCode"] = (
    train_cv_df["Period"]
    .map(period_mapping)
    .astype(int)
)

# สร้าง X และ y
X_cv_period = train_cv_df[features + ["PeriodCode"]].copy()
y_cv = train_cv_df[target_col].copy()

# ตรวจสอบข้อมูล
assert X_cv_period.shape[1] == 7
assert len(X_cv_period) == len(y_cv)
assert np.isfinite(X_cv_period.to_numpy(dtype=float)).all()

print("X_cv_period:", X_cv_period.shape)
print("y_cv:", y_cv.shape)
print("Features:", features)
print("Number of Periods:", len(period_categories))

print("\nPeriod Mapping:")
print(period_mapping)


X_cv_period: (94205, 7)
y_cv: (94205,)
Features: ['daily_return', 'momentum_20d', 'vol_z', 'VolRank', 'CorrRank', 'LiquidityRank']
Number of Periods: 18

Period Mapping:
{'2016_H1': 0, '2016_H2': 1, '2017_H1': 2, '2017_H2': 3, '2018_H1': 4, '2018_H2': 5, '2019_H1': 6, '2019_H2': 7, '2020_H1': 8, '2020_H2': 9, '2021_H1': 10, '2021_H2': 11, '2022_H1': 12, '2022_H2': 13, '2023_H1': 14, '2023_H2': 15, '2024_H1': 16, '2024_H2': 17}


In [83]:

# สร้างรายการวันที่ไม่ซ้ำและเรียงตามเวลา
unique_dates = np.array(
    sorted(train_cv_df["Target_Date"].unique())
)

# สร้าง Time Series Split
tscv = TimeSeriesSplit(n_splits=5)

cv_splits = []
fold_summary = []

# แบ่ง Fold ตามวันที่
for fold, (train_date_idx, val_date_idx) in enumerate(
    tscv.split(unique_dates), start=1
):
    train_dates = unique_dates[train_date_idx]
    val_dates = unique_dates[val_date_idx]

    # แปลงช่วงวันที่เป็น Row Indices
    train_idx = np.flatnonzero(
        train_cv_df["Target_Date"].isin(train_dates).to_numpy()
    )

    val_idx = np.flatnonzero(
        train_cv_df["Target_Date"].isin(val_dates).to_numpy()
    )

    # ตรวจสอบว่าไม่มีวันที่ซ้ำระหว่าง Train และ Validation
    assert len(
        set(train_dates).intersection(set(val_dates))
    ) == 0

    assert train_dates.max() < val_dates.min()

    cv_splits.append((train_idx, val_idx))

    fold_summary.append({
        "Fold": fold,
        "Train_Start": train_dates.min(),
        "Train_End": train_dates.max(),
        "Val_Start": val_dates.min(),
        "Val_End": val_dates.max(),
        "Train_Rows": len(train_idx),
        "Val_Rows": len(val_idx)
    })

# แสดงผลการแบ่ง Fold
fold_summary_df = pd.DataFrame(fold_summary)

print(fold_summary_df.to_string(index=False))

assert len(cv_splits) == 5

print("\nNumber of CV Folds:", len(cv_splits))


 Fold Train_Start  Train_End  Val_Start    Val_End  Train_Rows  Val_Rows
    1  2016-03-29 2017-09-15 2017-09-18 2019-02-26       14932     15113
    2  2016-03-29 2019-02-26 2019-02-27 2020-08-10       30045     15282
    3  2016-03-29 2020-08-10 2020-08-11 2022-01-28       45327     15944
    4  2016-03-29 2022-01-28 2022-01-31 2023-07-17       61271     16177
    5  2016-03-29 2023-07-17 2023-07-18 2024-12-30       77448     16757

Number of CV Folds: 5


In [84]:

# สร้าง Preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ("scale", StandardScaler(), features),
        ("period", "passthrough", ["PeriodCode"])
    ],
    remainder="drop"
)

print("Preprocessor created successfully")


Preprocessor created successfully


สมการกำหนดจำนวนตัวอย่างเป้าหมาย:

$$N_{Panic}^{*} = \max(N_{Panic}, \lceil rN_{Normal} \rceil)$$

$$N_{FOMO}^{*} = \max(N_{FOMO}, \lceil rN_{Normal} \rceil)$$


In [85]:

# สร้างฟังก์ชัน Period-aware SMOTE
def period_aware_smote_func(
    X,
    y,
    ratio=0.10,
    k_neighbors=3,
    random_state=42
):
    # แปลงข้อมูลเป็น NumPy Array
    X = np.asarray(X, dtype=float)
    y = np.asarray(y)

    # ตรวจสอบจำนวนคอลัมน์
    if X.ndim != 2 or X.shape[1] != 7:
        raise ValueError(
            f"Expected 7 columns, received {X.shape[1]}"
        )

    # แยก Features และ PeriodCode
    X_features = X[:, :6]
    period_codes = X[:, 6]

    X_parts = []
    y_parts = []

    # ทำ SMOTE แยกตาม Period
    for period_code in np.unique(period_codes):

        # เลือกเฉพาะข้อมูลใน Period ปัจจุบัน
        mask = period_codes == period_code

        X_period = X_features[mask]
        y_period = y[mask]

        # นับจำนวนแต่ละ Class
        counts = Counter(y_period)

        panic_before = counts.get(0, 0)
        fomo_before = counts.get(1, 0)
        normal_count = counts.get(2, 0)

        # คำนวณจำนวนเป้าหมาย
        target_panic = max(
            panic_before,
            ceil(ratio * normal_count)
        )

        target_fomo = max(
            fomo_before,
            ceil(ratio * normal_count)
        )

        # กำหนดจำนวนตัวอย่างที่จะเพิ่ม
        sampling_strategy = {}

        if target_panic > panic_before:
            sampling_strategy[0] = target_panic

        if target_fomo > fomo_before:
            sampling_strategy[1] = target_fomo

        # ทำ SMOTE เฉพาะภายใน Period นี้
        if sampling_strategy:

            for class_label in sampling_strategy:
                class_count = counts[class_label]

                if class_count <= k_neighbors:
                    raise ValueError(
                        f"Period {period_code}, "
                        f"Class {class_label}: "
                        f"Not enough samples for "
                        f"k_neighbors={k_neighbors}"
                    )

            sampler = SMOTE(
                sampling_strategy=sampling_strategy,
                k_neighbors=k_neighbors,
                random_state=random_state
            )

            X_period_resampled, y_period_resampled = (
                sampler.fit_resample(X_period, y_period)
            )

        else:
            X_period_resampled = X_period.copy()
            y_period_resampled = y_period.copy()

        # เก็บ PeriodCode กลับเข้าไป
        period_column = np.full(
            (len(X_period_resampled), 1),
            period_code
        )

        X_period_full = np.column_stack([
            X_period_resampled,
            period_column
        ])

        X_parts.append(X_period_full)
        y_parts.append(y_period_resampled)

    # รวมข้อมูลทุก Period
    X_resampled = np.vstack(X_parts)
    y_resampled = np.concatenate(y_parts)

    return X_resampled, y_resampled


# สร้าง FunctionSampler
def make_period_sampler(ratio, k_neighbors):

    return FunctionSampler(
        func=period_aware_smote_func,
        kw_args={
            "ratio": ratio,
            "k_neighbors": k_neighbors,
            "random_state": RANDOM_STATE
        },
        validate=False
    )

print("Period-aware SMOTE function created")


Period-aware SMOTE function created


In [86]:

# เลือก Fold แรก
train_idx, val_idx = cv_splits[0]

X_fold_train = X_cv_period.iloc[train_idx]
y_fold_train = y_cv.iloc[train_idx]

# Fit Preprocessor เฉพาะข้อมูล Training Fold
preprocessor_check = clone(preprocessor)

X_fold_processed = preprocessor_check.fit_transform(
    X_fold_train,
    y_fold_train
)

# กำหนด Ratio และ K สำหรับทดสอบ
test_ratio = 0.10
test_k = 3

# ทำ Period-aware SMOTE
sampler_check = make_period_sampler(
    ratio=test_ratio,
    k_neighbors=test_k
)

X_resampled, y_resampled = sampler_check.fit_resample(
    X_fold_processed,
    y_fold_train.to_numpy()
)

# ตรวจสอบจำนวนคอลัมน์
assert X_resampled.shape[1] == 7

# ตรวจสอบว่าจำนวนข้อมูลไม่ลดลง
assert len(X_resampled) >= len(X_fold_train)

# สร้างตารางนับจำนวน Class ก่อนและหลัง SMOTE
before_df = pd.DataFrame({
    "PeriodCode": X_fold_processed[:, 6].astype(int),
    "Target": y_fold_train.to_numpy()
})

after_df = pd.DataFrame({
    "PeriodCode": X_resampled[:, 6].astype(int),
    "Target": y_resampled
})

before_counts = (
    before_df.groupby(["PeriodCode", "Target"])
    .size()
    .unstack(fill_value=0)
)

after_counts = (
    after_df.groupby(["PeriodCode", "Target"])
    .size()
    .unstack(fill_value=0)
)

# จัดคอลัมน์ให้มี Class ครบ
before_counts = before_counts.reindex(
    columns=[0, 1, 2],
    fill_value=0
)

after_counts = after_counts.reindex(
    columns=[0, 1, 2],
    fill_value=0
)

print("Before SMOTE:")
print(before_counts)

print("\nAfter SMOTE:")
print(after_counts)

print("\nOriginal Rows:", len(X_fold_train))
print("Resampled Rows:", len(X_resampled))
print("Resampled Features:", X_resampled.shape[1])

# ตรวจสอบว่าไม่มีข้อมูลข้าม Period
assert set(after_counts.index).issubset(
    set(before_counts.index)
)

print("\nPeriod-aware SMOTE Test Passed")


Before SMOTE:
Target        0    1     2
PeriodCode                
0           102  139  2321
1           176  283  4625
2           214  282  4648
3            86  156  1900

After SMOTE:
Target        0    1     2
PeriodCode                
0           233  233  2321
1           463  463  4625
2           465  465  4648
3           190  190  1900

Original Rows: 14932
Resampled Rows: 16196
Resampled Features: 7

Period-aware SMOTE Test Passed


In [90]:

# สร้างฟังก์ชันเลือกเฉพาะ 6 Features
def select_model_features(X):

    X = np.asarray(X)

    return X[:, :6]


# สร้าง Feature Selector
feature_selector = FunctionTransformer(
    func=select_model_features,
    validate=False
)

print("Feature Selector created successfully")


Feature Selector created successfully


In [91]:

# สร้าง Pipeline สำหรับ Period-aware SMOTE
period_smote_pipeline = ImbPipeline([
    ("preprocessor", preprocessor),

    ("smote", make_period_sampler(
        ratio=0.10,
        k_neighbors=3
    )),

    ("feature_selector", feature_selector),

    ("model", LogisticRegression(
        C=10,
        solver="saga",
        l1_ratio=0,
        class_weight=None,
        max_iter=5000,
        random_state=RANDOM_STATE
    ))
])

print(period_smote_pipeline)


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('scale', StandardScaler(),
                                                  ['daily_return',
                                                   'momentum_20d', 'vol_z',
                                                   'VolRank', 'CorrRank',
                                                   'LiquidityRank']),
                                                 ('period', 'passthrough',
                                                  ['PeriodCode'])])),
                ('smote',
                 FunctionSampler(func=<function period_aware_smote_func at 0x000002A7C6BB9760>,
                                 kw_args={'k_neighbors': 3, 'random_state': 42,
                                          'ratio': 0.1},
                                 validate=False)),
                ('feature_selector',
                 FunctionTransformer(func=<function select_model_features at 0x000002A7C6BBBA60>)),
           

In [92]:

# เลือกข้อมูล Validation ของ Fold แรก
X_fold_val = X_cv_period.iloc[val_idx]
y_fold_val = y_cv.iloc[val_idx]

# สร้าง Pipeline สำหรับทดสอบ
test_pipeline = clone(period_smote_pipeline)

# ฝึกโมเดล
test_pipeline.fit(
    X_fold_train,
    y_fold_train
)

# ทำนาย Class และ Probability
y_pred_test = test_pipeline.predict(X_fold_val)

y_proba_test = test_pipeline.predict_proba(X_fold_val)

# ตรวจสอบผลลัพธ์
print("Training Rows:", len(X_fold_train))
print("Validation Rows:", len(X_fold_val))

print("Prediction Shape:", y_pred_test.shape)
print("Probability Shape:", y_proba_test.shape)

print(
    "Model Input Features:",
    test_pipeline.named_steps["model"].n_features_in_
)

# ตรวจสอบความถูกต้อง
assert y_pred_test.shape == (len(y_fold_val),)
assert y_proba_test.shape == (len(y_fold_val), 3)

assert test_pipeline.named_steps[
    "model"
].n_features_in_ == 6

assert np.isfinite(y_proba_test).all()

print("\nComplete Pipeline Test Passed")


Training Rows: 14932
Validation Rows: 15113
Prediction Shape: (15113,)
Probability Shape: (15113, 3)
Model Input Features: 6

Complete Pipeline Test Passed


In [93]:

# กำหนดพารามิเตอร์สำหรับ Grid Search
param_grid_period_smote = {
    "smote__kw_args": [
        {
            "ratio": ratio,
            "k_neighbors": k,
            "random_state": RANDOM_STATE
        }
        for ratio in [0.10, 0.25, 0.50]
        for k in [3, 5]
    ]
}

# กำหนด Metrics
scoring = {
    "macro_f1": "f1_macro",
    "balanced_accuracy": "balanced_accuracy",
    "neg_log_loss": "neg_log_loss"
}

# สร้าง Grid Search
period_smote_search = GridSearchCV(
    estimator=period_smote_pipeline,
    param_grid=param_grid_period_smote,
    scoring=scoring,
    refit="macro_f1",
    cv=cv_splits,
    n_jobs=-1,
    verbose=2,
    return_train_score=True,
    error_score="raise"
)

# เริ่ม Grid Search
period_smote_search.fit(
    X_cv_period,
    y_cv
)

print("\nBest Parameters:")
print(period_smote_search.best_params_)

print("\nBest Mean CV Macro F1:")
print(period_smote_search.best_score_)


Fitting 5 folds for each of 6 candidates, totalling 30 fits

Best Parameters:
{'smote__kw_args': {'ratio': 0.5, 'k_neighbors': 3, 'random_state': 42}}

Best Mean CV Macro F1:
0.3992077476345678


In [94]:

# แปลงผลลัพธ์เป็น DataFrame
period_smote_results = pd.DataFrame(
    period_smote_search.cv_results_
)

# แปลง Log Loss จากค่าติดลบ
period_smote_results["Mean_Log_Loss"] = (
    -period_smote_results["mean_test_neg_log_loss"]
)

# เลือกคอลัมน์สำหรับสรุปผล
period_smote_summary = period_smote_results[[
    "param_smote__kw_args",
    "mean_test_macro_f1",
    "std_test_macro_f1",
    "mean_test_balanced_accuracy",
    "Mean_Log_Loss",
    "mean_train_macro_f1",
    "rank_test_macro_f1"
]].copy()

# เปลี่ยนชื่อคอลัมน์
period_smote_summary = period_smote_summary.rename(
    columns={
        "param_smote__kw_args": "SMOTE_Parameters",
        "mean_test_macro_f1": "Mean_CV_Macro_F1",
        "std_test_macro_f1": "Std_CV_Macro_F1",
        "mean_test_balanced_accuracy": "Balanced_Accuracy",
        "mean_train_macro_f1": "Train_Macro_F1",
        "rank_test_macro_f1": "Rank"
    }
)

# เรียงตาม Macro F1
period_smote_summary = period_smote_summary.sort_values(
    "Mean_CV_Macro_F1",
    ascending=False
).reset_index(drop=True)

print(period_smote_summary.to_string(index=False))


                                     SMOTE_Parameters  Mean_CV_Macro_F1  Std_CV_Macro_F1  Balanced_Accuracy  Mean_Log_Loss  Train_Macro_F1  Rank
 {'ratio': 0.5, 'k_neighbors': 3, 'random_state': 42}          0.399208         0.010224           0.401003       0.728136        0.402707     1
 {'ratio': 0.5, 'k_neighbors': 5, 'random_state': 42}          0.399132         0.010020           0.401248       0.727615        0.402839     2
{'ratio': 0.25, 'k_neighbors': 5, 'random_state': 42}          0.362711         0.010783           0.357765       0.520401        0.368964     3
{'ratio': 0.25, 'k_neighbors': 3, 'random_state': 42}          0.362388         0.009706           0.357350       0.519968        0.368046     4
 {'ratio': 0.1, 'k_neighbors': 3, 'random_state': 42}          0.332522         0.004854           0.340682       0.389895        0.333850     5
 {'ratio': 0.1, 'k_neighbors': 5, 'random_state': 42}          0.331721         0.004625           0.340237       0.389520        

In [97]:

# เตรียม Validation Data
X_val_period = X_val.copy()

# กำหนด PeriodCode จาก Mapping ของ Training
X_val_period["PeriodCode"] = (
    validation_df["Period"]
    .map(period_mapping)
    .fillna(-1)
    .astype(int)
    .to_numpy()
)

# ตรวจสอบข้อมูล
assert X_val_period.shape[1] == 7
assert len(X_val_period) == len(y_val)
assert X_val_period["PeriodCode"].notna().all()

print("Validation PeriodCode Distribution:")
print(X_val_period["PeriodCode"].value_counts().sort_index())

print("\nUnseen Periods:")
print(
    sorted(
        set(validation_df["Period"].unique())
        - set(period_mapping.keys())
    )
)

print("\nValidation Data Ready")


Validation PeriodCode Distribution:
PeriodCode
-1     11777
 17       47
Name: count, dtype: int64

Unseen Periods:
['2025_H1', '2025_H2']

Validation Data Ready


In [98]:

# ใช้ Best Estimator จาก Grid Search
best_period_smote = period_smote_search.best_estimator_

# ทำนาย Validation
y_val_pred_period = best_period_smote.predict(X_val_period)

y_val_proba_period = best_period_smote.predict_proba(
    X_val_period
)

# คำนวณ Metrics
period_val_accuracy = accuracy_score(
    y_val, y_val_pred_period
)

period_val_macro_f1 = f1_score(
    y_val, y_val_pred_period, average="macro"
)

period_val_balanced_accuracy = balanced_accuracy_score(
    y_val, y_val_pred_period
)

period_val_log_loss = log_loss(
    y_val, y_val_proba_period, labels=[0, 1, 2]
)

# แสดงผล
print("Period-aware SMOTE - Validation 2025")
print("--------------------------------------")
print(f"Accuracy: {period_val_accuracy:.4f}")
print(f"Macro F1: {period_val_macro_f1:.4f}")
print(f"Balanced Accuracy: {period_val_balanced_accuracy:.4f}")
print(f"Log Loss: {period_val_log_loss:.4f}")

print("\nClassification Report:")
print(classification_report(
    y_val,
    y_val_pred_period,
    labels=[0, 1, 2],
    target_names=["Panic", "FOMO", "Normal"],
    digits=4,
    zero_division=0
))

print("\nConfusion Matrix:")
print(confusion_matrix(
    y_val,
    y_val_pred_period,
    labels=[0, 1, 2]
))


Period-aware SMOTE - Validation 2025
--------------------------------------
Accuracy: 0.8423
Macro F1: 0.4090
Balanced Accuracy: 0.4125
Log Loss: 0.7338

Classification Report:
              precision    recall  f1-score   support

       Panic     0.1583    0.2040    0.1783       544
        FOMO     0.1494    0.1167    0.1310       617
      Normal     0.9187    0.9168    0.9178     10663

    accuracy                         0.8423     11824
   macro avg     0.4088    0.4125    0.4090     11824
weighted avg     0.8436    0.8423    0.8427     11824


Confusion Matrix:
[[ 111   45  388]
 [  68   72  477]
 [ 522  365 9776]]


In [99]:
# กำหนด Final Model
final_model = grid_search.best_estimator_

# กำหนด Prediction Rule
final_prediction_rule = "Argmax"

# บันทึก Hyperparameters
final_params = grid_search.best_params_

# บันทึก Features
final_features = features.copy()

# ตรวจสอบโมเดล
print("Final Model:", final_model)
print("\nFinal Parameters:")
print(final_params)

print("\nFinal Features:")
print(final_features)

print("\nPrediction Rule:", final_prediction_rule)

# ตรวจสอบจำนวน Features
assert len(final_features) == 6
assert final_prediction_rule == "Argmax"

print("\nFinal Model Frozen")

Final Model: Pipeline(steps=[('scaler', StandardScaler()),
                ('model',
                 LogisticRegression(C=10, class_weight={0: 10, 1: 10, 2: 1},
                                    l1_ratio=0, max_iter=5000, random_state=42,
                                    solver='saga'))])

Final Parameters:
{'model__C': 10, 'model__class_weight': {0: 10, 1: 10, 2: 1}, 'model__l1_ratio': 0}

Final Features:
['daily_return', 'momentum_20d', 'vol_z', 'VolRank', 'CorrRank', 'LiquidityRank']

Prediction Rule: Argmax

Final Model Frozen


In [100]:
# ทำนาย Test 2026
y_test_pred = final_model.predict(X_test)

y_test_proba = final_model.predict_proba(X_test)

# คำนวณ Metrics
test_accuracy = accuracy_score(y_test, y_test_pred)

test_macro_f1 = f1_score(
    y_test,
    y_test_pred,
    average="macro"
)

test_balanced_accuracy = balanced_accuracy_score(
    y_test,
    y_test_pred
)

test_log_loss = log_loss(
    y_test,
    y_test_proba,
    labels=[0, 1, 2]
)

# แสดงผล Metrics
print("Final Test Results - 2026")
print("==========================")

print(f"Accuracy: {test_accuracy:.4f}")
print(f"Macro F1: {test_macro_f1:.4f}")
print(f"Balanced Accuracy: {test_balanced_accuracy:.4f}")
print(f"Log Loss: {test_log_loss:.4f}")

# แสดง Classification Report
print("\nClassification Report:")

print(classification_report(
    y_test,
    y_test_pred,
    labels=[0, 1, 2],
    target_names=["Panic", "FOMO", "Normal"],
    digits=4,
    zero_division=0
))

# แสดง Confusion Matrix
print("\nConfusion Matrix:")

print(confusion_matrix(
    y_test,
    y_test_pred,
    labels=[0, 1, 2]
))

# แสดง Prediction Distribution
print("\nPredicted Class Distribution:")

print(
    pd.Series(y_test_pred)
    .map({0: "Panic", 1: "FOMO", 2: "Normal"})
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Final Test Results - 2026
Accuracy: 0.8305
Macro F1: 0.3936
Balanced Accuracy: 0.3932
Log Loss: 0.7459

Classification Report:
              precision    recall  f1-score   support

       Panic     0.1321    0.0907    0.1075       386
        FOMO     0.1503    0.1800    0.1638       689
      Normal     0.9103    0.9089    0.9096      9361

    accuracy                         0.8305     10436
   macro avg     0.3976    0.3932    0.3936     10436
weighted avg     0.8314    0.8305    0.8307     10436


Confusion Matrix:
[[  35   60  291]
 [  18  124  547]
 [ 212  641 8508]]

Predicted Class Distribution:
Normal    89.56
FOMO       7.91
Panic      2.54
Name: proportion, dtype: float64


In [101]:

import joblib
import json
from pathlib import Path
from datetime import datetime

# กำหนดโฟลเดอร์สำหรับบันทึกโมเดล
model_dir = Path(
    r"D:\Project_Github\set_fear_index\models\Original_Softmax_Final"
)
model_dir.mkdir(parents=True, exist_ok=True)

# เลือกโมเดลที่ผ่านการคัดเลือกแล้ว
final_model = grid_search.best_estimator_

# กำหนดชื่อไฟล์
model_path = model_dir / "final_softmax_model.joblib"
metadata_path = model_dir / "final_softmax_metadata.json"
test_report_path = model_dir / "final_test_2026.json"

# บันทึกโมเดล
joblib.dump(final_model, model_path)

# เตรียมข้อมูล Metadata
metadata = {
    "model_name": "Original Softmax",
    "algorithm": "Multinomial Logistic Regression",
    "prediction_rule": "Argmax",
    "features": features,
    "class_mapping": {
        "0": "Panic",
        "1": "FOMO",
        "2": "Normal"
    },
    "best_parameters": grid_search.best_params_,
    "training_period": "2016-2024",
    "validation_period": "2025",
    "test_period": "2026",
    "training_samples": int(len(train_df)),
    "validation_samples": int(len(validation_df)),
    "test_samples": int(len(test_df)),
    "test_metrics": {
        "accuracy": 0.8305,
        "macro_f1": 0.3936,
        "balanced_accuracy": 0.3932,
        "log_loss": 0.7459
    },
    "created_at": datetime.now().isoformat()
}

# บันทึก Metadata
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=4, default=str)

# บันทึกผลการประเมิน Test
test_report = {
    "year": 2026,
    "accuracy": 0.8305,
    "macro_f1": 0.3936,
    "balanced_accuracy": 0.3932,
    "log_loss": 0.7459,
    "classification_report": {
        "Panic": {
            "precision": 0.1321,
            "recall": 0.0907,
            "f1": 0.1075,
            "support": 386
        },
        "FOMO": {
            "precision": 0.1503,
            "recall": 0.1800,
            "f1": 0.1638,
            "support": 689
        },
        "Normal": {
            "precision": 0.9103,
            "recall": 0.9089,
            "f1": 0.9096,
            "support": 9361
        }
    },
    "confusion_matrix": [
        [35, 60, 291],
        [18, 124, 547],
        [212, 641, 8508]
    ],
    "prediction_rule": "Argmax"
}

with open(test_report_path, "w", encoding="utf-8") as f:
    json.dump(test_report, f, ensure_ascii=False, indent=4)

# ตรวจสอบว่าโมเดลที่บันทึกสามารถโหลดกลับมาได้
loaded_model = joblib.load(model_path)

# ตรวจสอบผลทำนายหลังโหลด
prediction_check = loaded_model.predict(X_test)

assert (prediction_check == y_test_pred).all()

print("บันทึกโมเดลและตรวจสอบเรียบร้อย")
print("Model:", model_path)
print("Metadata:", metadata_path)
print("Test Report:", test_report_path)
print("Prediction consistency: Passed")


บันทึกโมเดลและตรวจสอบเรียบร้อย
Model: D:\Project_Github\set_fear_index\models\Original_Softmax_Final\final_softmax_model.joblib
Metadata: D:\Project_Github\set_fear_index\models\Original_Softmax_Final\final_softmax_metadata.json
Test Report: D:\Project_Github\set_fear_index\models\Original_Softmax_Final\final_test_2026.json
Prediction consistency: Passed
